*Auto-generated by [rep2nb](https://pypi.org/project/rep2nb/) — a tool that converts Python repos into executable notebooks.*

---

# Limestone Data Challenge 2026

> Full problem statement: [`data/limestone_data_challenge_2026.pdf`](data/limestone_data_challenge_2026.pdf)

![Problem Statement](data/challenge_pages.png)

A 3650×53 price matrix with ~50% missing values. Six columns are "index" columns (convex combinations of "farmer" columns). The challenge: identify indices, recover coefficients, complete the matrix, and build trading strategies.

### Challenge Overview

You run a bakery buying flour from *m* farmers at varying daily prices, published in a newspaper bulletin. Some prices are smudged (NaN). The bulletin also includes *n* index columns — convex combinations of farmer prices — but column identities are unknown.

| Problem | Task | Weight |
|---------|------|--------|
| 1a | Identify which columns are indices | 25% |
| 1b | Recover index decomposition coefficients | 5% |
| 2 | Fill all missing values in the matrix | 15% |
| 3 | Buy 100kg from NaN columns, minimize cost | 15% |
| 4 | Arbitrage: buy NaN, sell to index, maximize profit | 20% |
| 5 | Limit orders on NaN columns with price + quantity | 20% |

## Project Structure

```
Tower/
├── data/
│   ├── limestone_data_challenge_2026.data.csv   # raw input (3650 rows × 53 cols + time)
│   ├── limestone_data_challenge_2026.pdf        # full problem statement
│   └── challenge_pages.png                      # problem statement rendered as image
├── answers/                                      # generated outputs
│   ├── problem1a_answer.csv                      # column classification (farmer vs index)
│   ├── problem1b_answer.csv                      # index decomposition coefficients
│   └── problem2_answer.csv                       # fully completed matrix
├── problem-1_2/                                  # index detection + matrix completion
│   ├── pipeline.py                               # full pipeline (runs everything)
│   ├── candidates.py                             # phase 1: index detection via row-residual test
│   ├── coefficients.py                           # phase 2: NNLS coefficient recovery + constraint propagation
│   ├── matrix.py                                 # phase 3: iterative SVD completion (torch)
│   ├── trend.py                                  # periodic trend fitting (Lomb-Scargle + OLS harmonics)
│   ├── em.py                                     # standalone EM refinement (optional)
│   ├── intermediates/                            # cached results
│   │   ├── candidates.json                       # detected index/farmer column lists
│   │   ├── coefficients.json                     # raw decomposition weights (pre-EM)
│   │   ├── em_coefficients.json                  # refined decomposition weights (post-EM)
│   │   └── matrix.json                           # SVD rank sweep results + best rank
│   └── analysis/                                 # diagnostic plots
│       ├── residuals.png                         # bar chart: all columns ranked by convex-fit RMSE
│       ├── gaps.png                              # sorted RMSE + gap ratios between consecutive columns
│       ├── em_convergence.png                    # EM loop: verify RMSE, SVD RMSE, coef delta per iteration
│       └── trend_fits.png                        # per-column periodic trend fits vs observed data
├── problem-3_4/                                  # buying + arbitrage strategies
│   ├── buy.py                                    # trading_problem_3: buy 100kg cheapest
│   └── trade.py                                  # trading_problem_4: arbitrage
├── problem-5/                                    # limit-order buying
│   ├── pipeline.py                               # compute σ + backtest
│   ├── trade.py                                  # trading_problem_5: optimal bidding (classification inlined)
│   ├── compute_sigma.py                          # standalone σ computation (also available via pipeline)
│   └── intermediates/
│       └── sigma.json                            # precomputed per-column σ values
├── compact.py                                    # notebook generator (uses rep2nb)
├── limestone_data_challenge_2026.ipynb           # submission notebook (generated)
└── requirements.txt                              # Python dependencies
```

## Results

| Problem | Metric | Value |
|---------|--------|-------|
| **1a** | Indices detected | 6 identified (col_11, col_30, col_42, col_46, col_48, col_50) |
| **1b** | Coefficient sum | 1.0000 for all 6 indices |
| **1b** | Verify RMSE (co-observed) | < 0.01 for proven indices |
| **2** | SVD obs RMSE | 0.0181 (rank 47) |
| **3** | Cost vs oracle (3650 rows) | 56,169,176 / 56,169,176 — **100% oracle-hit rate** |
| **3** | Out-of-time cost/oracle | 106,095 / 100,927 — **5.1% above oracle** (7 synthetic rows) |
| **4** | Profit vs oracle (3650 rows) | 5,064,153 / 5,064,153 — **100% capture rate** |
| **4** | Out-of-time profit | 4,966 / 10,769 oracle (46%) |
| **5** | Fill rate (500 training rows) | 473/500 (**94.6%**) |
| **5** | Total score (profit) | 453,574 |
| **5** | Avg profit / day | 907.1 |

## Problems & Approaches

### Problem 1a — Index vs Farmer Classification

**Goal:** Identify which of the 53 columns are "index" columns (convex combinations of others).

**Approach:** For each column, greedily build a predictor set from its most correlated columns while maintaining sufficient co-observed rows. Fit a convex combination (non-negative weights summing to 1) on a train split, measure RMSE on a test split. True indices have near-zero test RMSE; farmers have high RMSE. A threshold of 3.5 cleanly separates the two groups.

**Result:** 6 index columns detected: `col_11`, `col_30`, `col_42`, `col_46`, `col_48`, `col_50`.

### Problem 1b — Coefficient Recovery

**Goal:** For each index column, recover the non-negative coefficients (summing to 1) expressing it as a convex combination of farmer columns.

**Approach:** Iterative rank-and-peel using multiple NNLS methods:
- Method A: NNLS on co-observed rows (original data)
- Method B: NNLS on constraint-filled data
- Method C: Greedy forward selection
- Method D: Farmer-only pool variants

Each method's result is verified via convex re-fit. Best result per column is accepted. An EM loop (SVD completion → re-regression) refines uncertain coefficients. Index-to-index dependencies are resolved via `distribute_through` to express everything in terms of pure farmer columns.

### Problem 2 — Matrix Completion

**Goal:** Fill all ~96,000 missing values.

**Approach:** Three-stage warm start followed by iterative SVD:
1. **Constraint propagation:** deterministically fill ~3,000 cells from proven index decompositions
2. **Trend model:** fit per-column periodic trends (Lomb-Scargle dominant period detection + OLS harmonic regression with up to 5 harmonics, BIC model selection for single vs double period)
3. **Iterative SVD:** rank-47 (= number of farmers) SVD completion using PyTorch, warm-started from steps 1+2

Final reconstruction enforces index constraints exactly and preserves all original observations.

### Problem 3 — Buy 100kg

**Goal:** Given a row with NaN prices, buy exactly 100kg from NaN-priced columns to minimize cost.

**Approach:**
- **Historical rows (t ≤ 3649):** look up predicted prices from the Problem 2 completed matrix
- **Out-of-time rows:** algebraic fills from decomposition constraints, then KNN (k=20) + low-rank SVD projection (rank 12), blended 50/50

Buy all 100kg from the single cheapest predicted NaN column.

### Problem 4 — Arbitrage

**Goal:** Buy from a NaN column and sell to an index column to maximize profit, up to 100kg.

**Approach:** Same price prediction as Problem 3. Find the cheapest NaN column (source) and the most expensive index column (destination). If dest > src, trade 100kg.

### Problem 5 — Limit-Order Buying

**Goal:** Place limit orders (price + quantity) on NaN columns. Orders fill only if bid ≥ true price. Score = Σ(qty × (median - bid) × I{fill}).

**Approach:**
- **Per-column uncertainty (σ):** estimated via leave-one-out cross-validation — hide one observed value, predict it via KNN+low-rank, record error. σ_j = std(errors). Precomputed and saved to `intermediates/sigma.json`.
- **Cell classification:** each NaN cell is tagged as algebraic (σ ≈ 0.02) or SVD-imputed (σ capped at 3.0 for historical, inflated 1.5× for out-of-time).
- **Optimal bid:** maximize E[profit/kg] = (median − bid) × Φ((bid − p̂) / σ) via bounded scalar optimization.
- **Allocation:** 100kg on best column for historical rows; spread across top 3 for out-of-time.

**Backtest result:** 94.6% fill rate, avg profit 907/day on 500 training rows.

## How to Run

### Prerequisites

```bash
pip install -r requirements.txt
```

### Step 1: Run the full pipeline (Problems 1 & 2)

```bash
cd problem-1_2
python3 pipeline.py --intermediates
```

This generates:
- `answers/problem1a_answer.csv` — column classifications
- `answers/problem1b_answer.csv` — decomposition coefficients
- `answers/problem2_answer.csv` — completed matrix
- `problem-1_2/intermediates/candidates.json` — detected index/farmer lists
- `problem-1_2/intermediates/coefficients.json` — raw decomposition weights (pre-EM)
- `problem-1_2/intermediates/em_coefficients.json` — refined weights (post-EM)
- `problem-1_2/intermediates/matrix.json` — SVD rank + obs RMSE
- `problem-1_2/analysis/*.png` — diagnostic plots

Runtime: ~2–5 minutes depending on hardware.

### Step 2: Run Problem 5 pipeline

```bash
cd problem-5
python3 pipeline.py                  # compute σ in memory, run backtest
python3 pipeline.py --intermediates  # also save intermediates/sigma.json
```

Without `--intermediates`: computes σ, builds cache, runs backtest — nothing written to disk.
With `--intermediates`: also saves `problem-5/intermediates/sigma.json` for standalone `trade.py` use.

Runtime: ~15 seconds.

### Step 3: Test trading strategies

```bash
# Problem 3 — buy strategy
cd problem-3_4
python3 buy.py

# Problem 4 — arbitrage
python3 trade.py

# Problem 5 — limit-order buying
cd ../problem-5
python3 trade.py
```

### Run everything from scratch

```bash
cd problem-1_2 && python3 pipeline.py --intermediates
cd ../problem-5 && python3 pipeline.py --intermediates
cd ../problem-3_4 && python3 buy.py && python3 trade.py
cd ../problem-5 && python3 trade.py
```

### Generate submission notebook

```bash
pip install rep2nb
python3 compact.py
```

Produces `limestone_data_challenge_2026.ipynb` — a fully executable notebook with correct dependency ordering, `!pip install` cell, and section isolation. Powered by [rep2nb](https://github.com/tyephoenix/rep2nb).

## Key Dependencies

| Package | Purpose |
|---------|---------|
| numpy | Linear algebra, array operations |
| pandas | Data I/O, DataFrames |
| scipy | NNLS, Lomb-Scargle, minimize_scalar, normal CDF |
| torch | GPU-accelerated iterative SVD (falls back to CPU) |
| rep2nb | Converts this repo into an executable notebook |


### Required Data Files

The following non-Python files from the repository should be present in the working directory for the notebook to run correctly:

- `data\challenge_pages.png`
- `data\limestone_data_challenge_2026.data.csv`
- `data\limestone_data_challenge_2026.pdf`
- `limestone_data_challenge_2026.ipynb`
- `problem-1_2\analysis\em_convergence.png`
- `problem-1_2\analysis\gaps.png`
- `problem-1_2\analysis\residuals.png`
- `problem-1_2\analysis\trend_fits.png`
- `problem-1_2\intermediates\candidates.json`
- `problem-1_2\intermediates\coefficients.json`
- `problem-1_2\intermediates\em_coefficients.json`
- `problem-1_2\intermediates\matrix.json`
- `problem-5\intermediates\sigma.json`


In [ ]:
import sys as _sys
!{_sys.executable} -m pip install -q matplotlib numpy pandas scipy torch
del _sys

## problem-1_2

In [ ]:
import os as _os; _os.makedirs('problem-1_2', exist_ok=True); _os.chdir('problem-1_2'); del _os

#### `problem-1_2/coefficients.py`

Phase 2 — Coefficient Recovery via NNLS.

Methods:
  A) NNLS on co-observed rows (adaptive pool, handles any k)
  B) NNLS on filled data (all rows where target observed, handles k=10+)
  C) Greedy forward selection with NNLS (fast)

Indices CAN appear as components of other indices. After proving,
distribute through to get the farmer-only decomposition.

In [ ]:
# === problem-1_2/coefficients.py ===

__file__ = 'coefficients.py'
import numpy as np
import pandas as pd
from scipy.optimize import minimize, nnls


def fit_convex(y, X):
    """Fit y ≈ X @ c with c >= 0, sum(c) = 1. Returns (coefs, rmse)."""
    n = X.shape[1]
    c0 = np.ones(n) / n
    res = minimize(
        lambda c: np.mean((y - X @ c) ** 2),
        c0, method="SLSQP",
        bounds=[(0, 1)] * n,
        constraints={"type": "eq", "fun": lambda c: c.sum() - 1.0},
        options={"maxiter": 5000, "ftol": 1e-15},
    )
    return res.x, np.sqrt(max(res.fun, 0))


def fit_nnls(y, X):
    """NNLS: y ≈ X @ c, c >= 0. Returns (coefs, rmse, weight_sum)."""
    c, _ = nnls(X, y)
    rmse = float(np.sqrt(np.mean((y - X @ c) ** 2)))
    return c, rmse, float(c.sum())


def _verify(idx_i, active_fi, data, mask_arr, min_rows=10):
    """Refit with convex constraint on co-observed data (original or filled)."""
    mask = mask_arr[:, idx_i].copy()
    for fi in active_fi:
        mask &= mask_arr[:, fi]
    rows = np.where(mask)[0]
    if len(rows) < min_rows:
        return None, float("inf"), len(rows)
    y = data[rows, idx_i]
    X = data[np.ix_(rows, active_fi)]
    coefs, rmse = fit_convex(y, X)
    return coefs, rmse, len(rows)


def _best_verify(idx_i, active_fi, arr, vmask, filled, fmask, min_rows=10):
    """Verify on both original and filled data, return the better result."""
    c1, r1, n1 = _verify(idx_i, active_fi, arr, vmask, min_rows)
    c2, r2, n2 = _verify(idx_i, active_fi, filled, fmask, min_rows)
    if c1 is not None and (c2 is None or r1 <= r2):
        return c1, r1, n1, "orig"
    if c2 is not None:
        return c2, r2, n2, "filled"
    return None, float("inf"), 0, None


# ── Method A: NNLS on co-observed rows ────────────────────────────────────

def nnls_coobs(idx_i, pool, arr, vmask, max_pool=30, min_rows=20):
    """
    Greedily build predictor set from most correlated columns while
    maintaining >= min_rows co-observed. Then fit NNLS in one shot.
    """
    corr_vals = []
    for fi in pool:
        both = vmask[:, idx_i] & vmask[:, fi]
        if both.sum() < 10:
            continue
        r = np.corrcoef(arr[both, idx_i], arr[both, fi])[0, 1]
        if not np.isnan(r):
            corr_vals.append((fi, abs(r)))
    corr_vals.sort(key=lambda x: -x[1])

    selected = []
    co_obs = vmask[:, idx_i].copy()
    for fi, _ in corr_vals:
        candidate_obs = co_obs & vmask[:, fi]
        if candidate_obs.sum() < min_rows:
            continue
        selected.append(fi)
        co_obs = candidate_obs
        if len(selected) >= max_pool:
            break

    if len(selected) < 2:
        return None, None, np.inf, 0.0, 0, 0

    rows = np.where(co_obs)[0]
    y = arr[rows, idx_i]
    X = arr[np.ix_(rows, selected)]
    coefs, rmse, w_sum = fit_nnls(y, X)

    active_fi = [fi for fi, w in zip(selected, coefs) if w > 1e-6]
    active_w = np.array([w for w in coefs if w > 1e-6])
    return active_fi, active_w, rmse, w_sum, len(rows), len(active_fi)


# ── Method B: NNLS on filled data ────────────────────────────────────────

def nnls_filled(idx_i, pool, filled, fmask, top_n=30):
    """
    NNLS using filled data. Remaining missing predictors get column means.
    Uses ALL rows where target is observed.
    """
    col_means = np.nanmean(filled, axis=0)
    rows = np.where(fmask[:, idx_i])[0]
    if len(rows) < 20:
        return None, None, np.inf, 0.0, 0, 0

    corr_vals = []
    for fi in pool:
        both = fmask[rows, fi]
        if both.sum() < 10:
            continue
        vals_y = filled[rows[both], idx_i]
        vals_x = filled[rows[both], fi]
        r = np.corrcoef(vals_y, vals_x)[0, 1]
        if not np.isnan(r):
            corr_vals.append((fi, abs(r)))
    corr_vals.sort(key=lambda x: -x[1])
    screened = [fi for fi, _ in corr_vals[:top_n]]

    if len(screened) < 2:
        return None, None, np.inf, 0.0, 0, 0

    y = filled[rows, idx_i]
    X = np.zeros((len(rows), len(screened)))
    for j, fi in enumerate(screened):
        obs = fmask[rows, fi]
        X[obs, j] = filled[rows[obs], fi]
        X[~obs, j] = col_means[fi]

    coefs, rmse, w_sum = fit_nnls(y, X)

    active_fi = [fi for fi, w in zip(screened, coefs) if w > 1e-6]
    active_w = np.array([w for w in coefs if w > 1e-6])
    return active_fi, active_w, rmse, w_sum, len(rows), len(active_fi)


# ── Method C: Greedy forward selection ────────────────────────────────────

def greedy_nnls(idx_i, pool, arr, vmask, max_k=20, min_rows=20):
    """Greedy forward selection using NNLS (fast)."""
    corr_vals = []
    for fi in pool:
        both = vmask[:, idx_i] & vmask[:, fi]
        if both.sum() < 10:
            continue
        r = np.corrcoef(arr[both, idx_i], arr[both, fi])[0, 1]
        if not np.isnan(r):
            corr_vals.append((fi, abs(r)))
    corr_vals.sort(key=lambda x: -x[1])
    pool_sorted = [fi for fi, _ in corr_vals]

    selected = []
    best_rmse = np.inf
    best_coefs = None

    for step in range(max_k):
        best_col = None
        best_step_rmse = np.inf
        best_step_coefs = None

        base_mask = vmask[:, idx_i].copy()
        for si in selected:
            base_mask &= vmask[:, si]

        for trial in pool_sorted:
            if trial in selected:
                continue
            mask = base_mask & vmask[:, trial]
            if mask.sum() < min_rows:
                continue
            cols_idx = selected + [trial]
            y = arr[mask, idx_i]
            X = arr[np.ix_(mask, cols_idx)]
            c, rmse, _ = fit_nnls(y, X)
            if rmse < best_step_rmse:
                best_step_rmse = rmse
                best_col = trial
                best_step_coefs = c

        if best_col is None:
            break
        selected.append(best_col)
        best_rmse = best_step_rmse
        best_coefs = best_step_coefs
        if best_rmse < 0.001:
            break

    if best_coefs is None:
        return None, None, np.inf
    active_fi = [fi for fi, w in zip(selected, best_coefs) if w > 1e-6]
    active_w = np.array([w for w in best_coefs if w > 1e-6])
    return active_fi, active_w, best_rmse


# ── Constraint propagation ────────────────────────────────────────────────

def fill_from_known(arr, vmask, decompositions, cols, max_passes=10,
                    verbose=True):
    """
    Use proven decompositions to fill missing values deterministically.
    """
    filled = arr.copy()
    fmask = vmask.copy()
    total_filled = 0

    for pass_num in range(max_passes):
        new_fills = 0
        for idx_col, dec in decompositions.items():
            idx_i = cols.index(idx_col)
            fi_list = dec["farmer_idxs"]
            coefs = np.array(dec["coefs"], dtype=float)

            idx_obs = fmask[:, idx_i]
            for row in np.where(idx_obs)[0]:
                farmer_obs = fmask[row, fi_list]
                n_missing = (~farmer_obs).sum()
                if n_missing != 1:
                    continue
                missing_j = int(np.where(~farmer_obs)[0][0])
                missing_fi = fi_list[missing_j]
                w_missing = coefs[missing_j]
                if w_missing < 1e-8:
                    continue
                known_sum = sum(coefs[j] * filled[row, fi_list[j]]
                                for j in range(len(fi_list)) if j != missing_j)
                filled[row, missing_fi] = (filled[row, idx_i] - known_sum) / w_missing
                fmask[row, missing_fi] = True
                new_fills += 1

            for row in np.where(~idx_obs)[0]:
                if fmask[row, fi_list].all():
                    filled[row, idx_i] = sum(coefs[j] * filled[row, fi_list[j]]
                                             for j in range(len(fi_list)))
                    fmask[row, idx_i] = True
                    new_fills += 1

        total_filled += new_fills
        if new_fills == 0:
            break

    if verbose and total_filled > 0:
        obs_before = vmask.sum()
        obs_after = fmask.sum()
        print(f"  [Fill] {total_filled} values filled in {pass_num+1} passes "
              f"({obs_before} → {obs_after} observed)")
    return filled, fmask


# ── Distribute through: expand index components to farmers ────────────────

def distribute_through(decompositions, cols):
    """
    For each proven index, if its decomposition includes another proven index,
    substitute that index's decomposition to get all-farmer weights.
    Repeat until no index components remain.
    """
    proven = {c for c, d in decompositions.items() if d.get("proven")}
    expanded = {}

    for idx_col, dec in decompositions.items():
        if not dec.get("proven"):
            continue
        weights = {}
        for fi, w in zip(dec["farmer_idxs"], dec["coefs"]):
            col_name = cols[fi]
            weights[col_name] = weights.get(col_name, 0.0) + float(w)

        changed = True
        while changed:
            changed = False
            new_weights = {}
            for comp_col, w in weights.items():
                if comp_col in proven and comp_col != idx_col and comp_col in decompositions:
                    sub = decompositions[comp_col]
                    for sfi, sw in zip(sub["farmer_idxs"], sub["coefs"]):
                        sub_name = cols[sfi]
                        new_weights[sub_name] = new_weights.get(sub_name, 0.0) + w * float(sw)
                    changed = True
                else:
                    new_weights[comp_col] = new_weights.get(comp_col, 0.0) + w
            weights = new_weights

        farmer_names = sorted(weights.keys(), key=lambda c: -weights[c])
        farmer_idxs = [cols.index(c) for c in farmer_names if weights[c] > 1e-6]
        farmer_coefs = np.array([weights[c] for c in farmer_names if weights[c] > 1e-6])

        expanded[idx_col] = {
            "farmer_idxs": farmer_idxs,
            "coefs": farmer_coefs,
            "farmer_names": [cols[fi] for fi in farmer_idxs],
        }

    return expanded


# ── Re-regression helpers (for EM loop in pipeline) ───────────────────────

def refit_weights(idx_i, fixed_farmer_idxs, completed):
    """Re-fit convex weights on completed data with a fixed set of farmers."""
    y = completed[:, idx_i]
    X = completed[:, fixed_farmer_idxs]
    coefs, rmse = fit_convex(y, X)
    return list(fixed_farmer_idxs), coefs, rmse


def reregress_on_completed(idx_i, farmer_idxs, completed, cols, top_k=25):
    """Re-regress on SVD-completed data using greedy + NNLS."""
    y = completed[:, idx_i]
    corr_vals = []
    for fi in farmer_idxs:
        r = np.corrcoef(y, completed[:, fi])[0, 1]
        corr_vals.append((fi, abs(r) if not np.isnan(r) else 0.0))
    pool = [fi for fi, _ in sorted(corr_vals, key=lambda x: -x[1])[:top_k]]

    X = completed[:, pool]
    coefs, rmse, w_sum = fit_nnls(y, X)
    active_fi = [fi for fi, w in zip(pool, coefs) if w > 1e-6]
    active_w = np.array([w for w in coefs if w > 1e-6])

    if not active_fi:
        return pool[:3], np.ones(3) / 3, rmse

    y_v = completed[:, idx_i]
    X_v = completed[:, active_fi]
    c_final, rmse_final = fit_convex(y_v, X_v)
    return active_fi, c_final, rmse_final


# ── Orchestrator ─────────────────────────────────────────────────────────

PROVEN_THRESHOLD = 0.01

def _try_one_candidate(idx_col, idx_i, all_others, arr, vmask, filled, fmask,
                       cols, proven_idxs=None, verbose=True):
    """
    Try all methods for a single candidate, return best.
    proven_idxs: set of column indices already proven as indices.
    Methods A-C use full pool; Method D uses farmer-only pool (excludes proven).
    """
    best_method = None
    best_fi = None
    best_c = None
    best_score = np.inf
    if proven_idxs is None:
        proven_idxs = set()

    def _run_method(label, active_fi, active_w=None, raw_rmse=None,
                    raw_sum=None, raw_rows=None, raw_k=None):
        """Verify a method result on both original and filled, update best."""
        nonlocal best_method, best_fi, best_c, best_score
        if active_fi is None:
            return
        vc, vr, vn, vsrc = _best_verify(
            idx_i, active_fi, arr, vmask, filled, fmask)
        if verbose:
            extra = ""
            if raw_rmse is not None:
                extra += f"RMSE={raw_rmse:.6f}"
            if raw_sum is not None:
                extra += f", sum={raw_sum:.4f}"
            if raw_rows is not None:
                extra += f", rows={raw_rows}"
            if raw_k is not None:
                extra = f"pool→{raw_k} active, " + extra
            if extra:
                print(f"  [{label}] {extra}")
            if vc is not None:
                print(f"    → verify({vsrc}): RMSE={vr:.8f}, rows={vn}")
        if vc is not None and vr < best_score:
            best_method, best_fi, best_c, best_score = (
                label, active_fi, vc, vr)

    # Method A: NNLS on co-observed rows (original data)
    a_fi, a_w, a_rmse, a_sum, a_rows, a_k = nnls_coobs(
        idx_i, all_others, arr, vmask)
    _run_method("nnls_coobs", a_fi, a_w, a_rmse, a_sum, a_rows, a_k)

    # Method B: NNLS on filled data
    b_fi, b_w, b_rmse, b_sum, b_rows, b_k = nnls_filled(
        idx_i, all_others, filled, fmask)
    _run_method("nnls_filled", b_fi, b_w, b_rmse, b_sum, b_rows, b_k)

    # Method C: Greedy forward selection (original data, full pool)
    c_fi, c_w, c_rmse = greedy_nnls(idx_i, all_others, arr, vmask)
    if c_fi is not None:
        _run_method("greedy", c_fi, c_w, c_rmse, raw_k=len(c_fi))

    # Method D: Greedy on farmer-only pool (skip proven indices)
    if proven_idxs:
        farmer_pool = [j for j in all_others if j not in proven_idxs]

        d_fi, d_w, d_rmse = greedy_nnls(idx_i, farmer_pool, arr, vmask)
        if d_fi is not None:
            _run_method("greedy_farmers", d_fi, d_w, d_rmse,
                        raw_k=len(d_fi))

        # Greedy on farmer-only pool with FILLED data
        e_fi, e_w, e_rmse = greedy_nnls(idx_i, farmer_pool, filled, fmask)
        if e_fi is not None:
            _run_method("greedy_farmers_filled", e_fi, e_w, e_rmse,
                        raw_k=len(e_fi))

        # NNLS-coobs on farmer-only pool
        f_fi, f_w, f_rmse, f_sum, f_rows, f_k = nnls_coobs(
            idx_i, farmer_pool, arr, vmask)
        _run_method("nnls_coobs_farmers", f_fi, f_w, f_rmse,
                    f_sum, f_rows, f_k)

    if verbose:
        print(f"  >>> Best: {best_method} (RMSE={best_score:.8f})")
        if best_fi is not None:
            for fi, w in sorted(zip(best_fi, best_c), key=lambda x: -x[1]):
                if w > 0.005:
                    print(f"      {cols[fi]}: {w:.6f}")

    return best_method, best_fi, best_c, best_score


def recover_coefficients(data, arr, vmask, cols,
                         index_cols, index_idxs, farmer_cols, farmer_idxs,
                         max_passes=20, accept_threshold=0.6, verbose=True):
    """
    Iterative rank-and-peel coefficient recovery.

    Pool = ALL other columns (indices allowed — distribute through later).
    Fill from ALL accepted decompositions each pass.
    """
    D = len(cols)

    if verbose:
        print(f"\n{'='*70}")
        print(f"  PHASE 2: Iterative Coefficient Recovery (NNLS)")
        print(f"  Rank-and-peel | proven<{PROVEN_THRESHOLD} | tentative<{accept_threshold}")
        print(f"{'='*70}")

    decompositions = {}
    accepted_set = set()
    remaining = list(zip(index_cols, index_idxs))

    for pass_num in range(1, max_passes + 1):
        if not remaining:
            break

        # Fill from ALL accepted decompositions
        if accepted_set:
            all_decs = {c: decompositions[c] for c in accepted_set}
            filled, fmask = fill_from_known(
                arr, vmask, all_decs, cols, verbose=verbose)
        else:
            filled, fmask = arr.copy(), vmask.copy()

        if verbose:
            obs_rate = fmask.sum() / fmask.size
            print(f"\n  ══ Pass {pass_num} ({len(remaining)} candidates, "
                  f"{obs_rate:.1%} observed) ══")

        # Build set of proven column indices for farmer-only pool
        proven_idxs = set()
        for c in accepted_set:
            ci = cols.index(c)
            proven_idxs.add(ci)

        # Score all remaining (pool = ALL other columns, never exclude)
        scores = []
        for idx_col, idx_i in remaining:
            if verbose:
                print(f"\n  --- {idx_col} ---")

            all_others = [j for j in range(D) if j != idx_i]
            method, fi, coefs, score = _try_one_candidate(
                idx_col, idx_i, all_others, arr, vmask,
                filled, fmask, cols,
                proven_idxs=proven_idxs, verbose=verbose)
            scores.append((idx_col, idx_i, method, fi, coefs, score))

        scores.sort(key=lambda x: x[5])

        if verbose:
            print(f"\n  Ranking:")
            for idx_col, _, _, _, _, score in scores:
                tag = ("PROVEN" if score < PROVEN_THRESHOLD else
                       "tentative" if score < accept_threshold else "reject")
                print(f"    {idx_col}: RMSE={score:.6f} [{tag}]")

        # Accept: all proven, or single best tentative
        proven = [(c, i, m, f, co, s) for c, i, m, f, co, s in scores
                  if s < PROVEN_THRESHOLD]
        new_accepted = []

        if proven:
            for idx_col, idx_i, method, fi, coefs, score in proven:
                decompositions[idx_col] = {
                    "method": method,
                    "farmer_idxs": fi if fi is not None else [],
                    "coefs": coefs if coefs is not None else np.array([]),
                    "proven": True,
                }
                accepted_set.add(idx_col)
                new_accepted.append(idx_col)
                if verbose:
                    print(f"  ✓ PROVEN: {idx_col} (RMSE={score:.8f})")
        else:
            best = scores[0]
            idx_col, idx_i, method, fi, coefs, score = best
            if score < accept_threshold and fi is not None:
                decompositions[idx_col] = {
                    "method": method,
                    "farmer_idxs": fi,
                    "coefs": coefs,
                    "proven": False,
                }
                accepted_set.add(idx_col)
                new_accepted.append(idx_col)
                if verbose:
                    print(f"  ~ TENTATIVE: {idx_col} (RMSE={score:.6f})")
            else:
                if verbose:
                    print(f"  No candidate below threshold — stopping.")
                break

        remaining = [(c, i) for c, i, _, _, _, _ in scores
                     if c not in accepted_set]

        if verbose:
            print(f"  Pass {pass_num}: accepted {new_accepted}, "
                  f"{len(remaining)} remaining")

    # Distribute through: expand index components to pure farmers
    expanded = distribute_through(decompositions, cols)
    if verbose and expanded:
        print(f"\n  ── Distributed to farmers ──")
        for idx_col, exp in expanded.items():
            names = [f"{cols[fi]}:{w:.4f}"
                     for fi, w in zip(exp["farmer_idxs"], exp["coefs"])]
            print(f"    {idx_col} = {' + '.join(names)}")

    # Final fill with all accepted decompositions
    if accepted_set:
        all_decs = {c: decompositions[c] for c in accepted_set}
        filled, fmask = fill_from_known(arr, vmask, all_decs, cols,
                                         verbose=verbose)
    else:
        filled, fmask = arr.copy(), vmask.copy()

    confirmed = [c for c in index_cols if c in accepted_set]
    failed = [c for c in index_cols if c not in accepted_set]
    if verbose:
        print(f"\n  ┌─────────────────────────────────────────┐")
        print(f"  │  Accepted indices: {confirmed}")
        if failed:
            print(f"  │  Reclassified as farmers: {failed}")
        obs_rate = fmask.sum() / fmask.size
        print(f"  │  Filled matrix: {obs_rate:.1%} observed")
        print(f"  └─────────────────────────────────────────┘")

    return decompositions, filled, fmask


# ── Serialization ─────────────────────────────────────────────────────────

def save_coefficients(decompositions, index_cols, index_idxs,
                      farmer_cols, farmer_idxs, cols,
                      path="intermediates/coefficients.json"):
    """Serialize decompositions to JSON."""
    import json, os
    os.makedirs(os.path.dirname(path), exist_ok=True)
    out = {
        "index_cols": index_cols,
        "index_idxs": index_idxs,
        "farmer_cols": farmer_cols,
        "farmer_idxs": farmer_idxs,
        "cols": list(cols),
        "decompositions": {},
    }
    for idx_col, dec in decompositions.items():
        out["decompositions"][idx_col] = {
            "method": dec["method"],
            "farmer_idxs": [int(x) for x in dec["farmer_idxs"]],
            "coefs": [float(x) for x in dec["coefs"]],
        }
    with open(path, "w") as f:
        json.dump(out, f, indent=2)
    print(f"  Saved {path}")


def load_coefficients(path="intermediates/coefficients.json"):
    """Load decompositions from JSON."""
    import json
    with open(path) as f:
        raw = json.load(f)
    for dec in raw["decompositions"].values():
        dec["coefs"] = np.array(dec["coefs"])
    return raw


# ── Standalone entry point ────────────────────────────────────────────────

# --- rep2nb: register module so imports resolve ---
import types as _types, sys as _sys
_mod = _sys.modules.get('coefficients') or _types.ModuleType('coefficients')
for _n in ['np', 'pd', 'minimize', 'nnls', 'fit_convex', 'fit_nnls', '_verify', '_best_verify', 'nnls_coobs', 'nnls_filled', 'greedy_nnls', 'fill_from_known', 'distribute_through', 'refit_weights', 'reregress_on_completed', 'PROVEN_THRESHOLD', '_try_one_candidate', 'recover_coefficients', 'save_coefficients', 'load_coefficients']:
    if _n in globals(): setattr(_mod, _n, globals()[_n])
_sys.modules['coefficients'] = _mod
del _types, _sys, _mod


#### `problem-1_2/matrix.py`

Phase 3 — Matrix Completion & Rebuild.
SVD-based iterative completion, then optional re-regression of uncertain
index coefficients on the completed data.

In [ ]:
# === problem-1_2/matrix.py ===

__file__ = 'matrix.py'
import numpy as np
import pandas as pd


def iterative_svd_complete(data_np, mask_np, rank,
                           max_iter=500, rel_tol=1e-4, warm_start=None):
    """
    Iterative SVD using torch (GPU if available).

    Parameters
    ----------
    warm_start : np.ndarray or None
        If provided, use this as the initial fill for missing values
        instead of column means. Must be same shape as data_np with no NaN.

    Returns (completed, obs_rmse).
    """
    import torch

    if torch.cuda.is_available():
        dev = torch.device("cuda")
        dtype = torch.float64
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        dev = torch.device("mps")
        dtype = torch.float32
    else:
        dev = torch.device("cpu")
        dtype = torch.float64

    if warm_start is not None:
        X_np = warm_start.copy()
    else:
        col_means = np.nanmean(data_np, axis=0)
        X_np = data_np.copy()
        X_np[~mask_np] = np.take(col_means, np.where(~mask_np)[1])

    X = torch.tensor(X_np, dtype=dtype, device=dev)
    mask = torch.tensor(mask_np, dtype=torch.bool, device=dev)
    data_obs = torch.tensor(np.nan_to_num(data_np, nan=0.0),
                            dtype=dtype, device=dev)

    prev_rmse = float("inf")
    for it in range(max_iter):
        U, S, Vt = torch.linalg.svd(X, full_matrices=False)
        recon = U[:, :rank] @ torch.diag(S[:rank]) @ Vt[:rank, :]
        X = torch.where(mask, data_obs, recon)

        diff = (data_obs - recon) * mask
        rmse = torch.sqrt((diff ** 2).sum() / mask.sum()).item()
        if abs(prev_rmse - rmse) / (rmse + 1e-15) < rel_tol:
            break
        prev_rmse = rmse

    return X.cpu().numpy(), rmse


def rank_sweep(arr, vmask, ranks=None, max_iter=300, rel_tol=1e-5,
               verbose=True):
    """
    Sweep over ranks, return list of (rank, obs_rmse).
    """
    if ranks is None:
        ranks = list(range(2, 21)) + list(range(25, 46, 5))

    if verbose:
        print(f"  Rank sweep (full {arr.shape[1]}-col matrix)...")
        print(f"  {'Rank':>5} {'ObsRMSE':>10}")

    results = []
    for rank in ranks:
        _, obs_rmse = iterative_svd_complete(
            arr, vmask, rank, max_iter=max_iter, rel_tol=rel_tol)
        results.append((rank, obs_rmse))
        if verbose:
            print(f"  {rank:5d} {obs_rmse:10.6f}")

    return results


def pick_best_rank(rank_results, rmse_threshold=0.1):
    """Pick first rank below threshold, else lowest RMSE."""
    good = [(r, e) for r, e in rank_results if e < rmse_threshold]
    if good:
        return good[0][0], good[0][1]
    best = min(rank_results, key=lambda x: x[1])
    return best[0], best[1]


def svd_best_rank(arr, vmask, ranks=None, verbose=True):
    """Rank sweep → pick best → complete at that rank. Returns (rank, completed, obs_rmse)."""
    rank_results = rank_sweep(arr, vmask, ranks=ranks, verbose=verbose)
    best_rank, _ = pick_best_rank(rank_results)
    if verbose:
        print(f"  Selected rank={best_rank}")
    completed, obs_rmse = iterative_svd_complete(
        arr, vmask, best_rank, max_iter=500, rel_tol=1e-6)
    return best_rank, completed, obs_rmse


def complete_matrix(arr, vmask, cols, decompositions,
                    filled=None, fmask=None, rank=None, verbose=True):
    """
    SVD completion + reconstruct index columns from decompositions.
    If filled/fmask provided, uses them as warm start.
    Returns (completed, obs_rmse).
    """
    if rank is None:
        n_indices = len(decompositions)
        rank = len(cols) - n_indices

    if verbose:
        print(f"\n  Matrix completion: rank={rank}, "
              f"{len(decompositions)} index columns")

    if filled is None or fmask is None:
        from coefficients import fill_from_known
        filled, fmask = fill_from_known(arr, vmask, decompositions, cols,
                                         verbose=verbose)

    completed, obs_rmse = iterative_svd_complete(filled, fmask, rank)
    if verbose:
        print(f"  SVD obs RMSE: {obs_rmse:.6f}")

    for idx_col, dec in decompositions.items():
        idx_i = cols.index(idx_col)
        completed[:, idx_i] = (completed[:, dec["farmer_idxs"]]
                               @ dec["coefs"])

    completed[vmask] = arr[vmask]
    return completed, obs_rmse


def save_matrix_result(decompositions, best_rank, final_rmse, rank_results,
                       cols, path="intermediates/matrix.json"):
    """Serialize matrix completion metadata to JSON."""
    import json, os
    os.makedirs(os.path.dirname(path), exist_ok=True)
    out = {
        "best_rank": int(best_rank),
        "final_rmse": float(final_rmse),
        "rank_results": [[int(r), float(e)] for r, e in rank_results],
        "decompositions": {},
    }
    for idx_col, dec in decompositions.items():
        out["decompositions"][idx_col] = {
            "method": dec["method"],
            "farmer_idxs": [int(x) for x in dec["farmer_idxs"]],
            "coefs": [float(x) for x in dec["coefs"]],
        }
    with open(path, "w") as f:
        json.dump(out, f, indent=2)
    print(f"  Saved {path}")


# ── Standalone entry point ────────────────────────────────────────────────

# --- rep2nb: register module so imports resolve ---
import types as _types, sys as _sys
_mod = _sys.modules.get('matrix') or _types.ModuleType('matrix')
for _n in ['np', 'pd', 'iterative_svd_complete', 'rank_sweep', 'pick_best_rank', 'svd_best_rank', 'complete_matrix', 'save_matrix_result']:
    if _n in globals(): setattr(_mod, _n, globals()[_n])
_sys.modules['matrix'] = _mod
del _types, _sys, _mod


#### `problem-1_2/trend.py`

Per-column periodic trend model.

Finds dominant period via Lomb-Scargle periodogram (O(n log n)),
then fits harmonics via OLS. No grid search needed.

In [ ]:
# === problem-1_2/trend.py ===

__file__ = 'trend.py'
import numpy as np
from scipy.signal import lombscargle

N_HARM = 5


def _find_periods(t, y, n_periods=2):
    """Find top dominant periods using Lomb-Scargle periodogram."""
    # Detrend
    slope, intercept = np.polyfit(t, y, 1)
    y_detrend = y - (slope * t + intercept)

    # Candidate angular frequencies
    min_period, max_period = 30.0, 300.0
    freqs = np.linspace(2 * np.pi / max_period, 2 * np.pi / min_period, 2000)

    power = lombscargle(t, y_detrend, freqs, normalize=True)

    periods_found = []
    for _ in range(n_periods):
        if len(power) == 0:
            break
        idx = np.argmax(power)
        P = 2 * np.pi / freqs[idx]
        periods_found.append(P)
        # Zero out around this peak and its harmonics
        for mult in [0.5, 1.0, 2.0]:
            mask = np.abs(freqs - mult * freqs[idx]) < freqs[idx] * 0.15
            power[mask] = 0.0

    return periods_found


def _build_basis(t, P, n_harm=N_HARM):
    """Design matrix for one period: sin/cos pairs + trend + intercept."""
    n = len(t)
    X = np.empty((n, 2 * n_harm + 2))
    for k in range(1, n_harm + 1):
        arg = 2 * np.pi * k * t / P
        X[:, 2*(k-1)] = np.sin(arg)
        X[:, 2*(k-1)+1] = np.cos(arg)
    X[:, -2] = t
    X[:, -1] = 1.0
    return X


def fit_column(t_obs, y_obs):
    """
    Fit harmonic trend model using Lomb-Scargle + OLS.
    Returns dict: P1, P2, coefs, n_harm, rmse
    """
    n = len(t_obs)
    if n < 10:
        mu = np.mean(y_obs) if n > 0 else 160.0
        p = 2 * N_HARM + 2
        c = np.zeros(p)
        c[-1] = mu
        return {"P1": 79.0, "P2": None, "coefs": c,
                "n_harm": N_HARM, "rmse": float("inf")}

    t = t_obs.astype(np.float64)
    y = y_obs.astype(np.float64)

    periods = _find_periods(t, y, n_periods=2)
    P1 = periods[0] if periods else 79.0

    # Single-period fit
    X1 = _build_basis(t, P1)
    coefs1, _, _, _ = np.linalg.lstsq(X1, y, rcond=None)
    sse1 = np.sum((y - X1 @ coefs1) ** 2)

    # Try double-period if we found a second
    P2 = None
    coefs_final = coefs1
    sse_final = sse1
    if len(periods) >= 2:
        P2_cand = periods[1]
        X_joint = np.column_stack([
            _build_basis(t, P1)[:, :-2],
            _build_basis(t, P2_cand),
        ])
        coefs_joint, _, _, _ = np.linalg.lstsq(X_joint, y, rcond=None)
        sse_joint = np.sum((y - X_joint @ coefs_joint) ** 2)

        p1_params = 2 * N_HARM + 2
        p2_params = 4 * N_HARM + 2
        bic1 = n * np.log(sse1 / n + 1e-15) + p1_params * np.log(n)
        bic2 = n * np.log(sse_joint / n + 1e-15) + p2_params * np.log(n)

        if bic2 < bic1 - 1.0:
            P2 = P2_cand
            coefs_final = coefs_joint
            sse_final = sse_joint

    rmse = np.sqrt(sse_final / n)
    return {"P1": float(P1), "P2": float(P2) if P2 else None,
            "coefs": coefs_final, "n_harm": N_HARM, "rmse": rmse}


def trend_model(t, params):
    """Evaluate fitted model at times t."""
    P1 = params["P1"]
    P2 = params["P2"]
    coefs = params["coefs"]
    nh = params["n_harm"]

    if P2 is None:
        X = _build_basis(t, P1, nh)
    else:
        X = np.column_stack([
            _build_basis(t, P1, nh)[:, :-2],
            _build_basis(t, P2, nh),
        ])
    return X @ coefs


def fit_all_columns(arr, vmask, t_all, cols, verbose=True):
    """Fit trend model for every column."""
    fits = {}
    for j, col in enumerate(cols):
        obs = vmask[:, j]
        t_obs = t_all[obs].astype(float)
        y_obs = arr[obs, j]
        fits[col] = fit_column(t_obs, y_obs)
        if verbose:
            f = fits[col]
            p2_str = f"P2={f['P2']:.1f}" if f['P2'] else "single"
            print(f"  {col}: P1={f['P1']:.1f} {p2_str:>12s}  "
                  f"rmse={f['rmse']:.2f}")
    return fits


def build_warm_start(arr, vmask, t_all, cols, fits):
    """Fill NaN cells with fitted trend model. Observed values preserved."""
    T, D = arr.shape
    warm = arr.copy()
    t_float = t_all.astype(float)
    for j in range(D):
        missing = ~vmask[:, j]
        if missing.any():
            warm[missing, j] = trend_model(t_float[missing], fits[cols[j]])
    return warm

# --- rep2nb: register module so imports resolve ---
import types as _types, sys as _sys
_mod = _sys.modules.get('trend') or _types.ModuleType('trend')
for _n in ['np', 'lombscargle', 'N_HARM', '_find_periods', '_build_basis', 'fit_column', 'trend_model', 'fit_all_columns', 'build_warm_start']:
    if _n in globals(): setattr(_mod, _n, globals()[_n])
_sys.modules['trend'] = _mod
del _types, _sys, _mod


#### `problem-1_2/candidates.py`

Phase 1 — Row-Residual Index Detection.

For each column, measure how well it can be predicted as a convex combination
of its top-K correlated columns on co-observed rows (train/test split).
True indices have near-zero test RMSE; farmers have positive RMSE.

In [ ]:
# === problem-1_2/candidates.py ===

__file__ = 'candidates.py'
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from coefficients import fit_convex


DEFAULT_MIN_ROWS = 30
DEFAULT_N_SPLITS = 15
DEFAULT_RMSE_THRESHOLD = 3.5


def pairwise_abs_corr(arr, vmask, min_obs=50):
    """Pairwise absolute Pearson correlation on co-observed rows."""
    D = arr.shape[1]
    corr = np.full((D, D), np.nan)
    for i in range(D):
        for j in range(i + 1, D):
            both = vmask[:, i] & vmask[:, j]
            if both.sum() < min_obs:
                continue
            r = np.corrcoef(arr[both, i], arr[both, j])[0, 1]
            corr[i, j] = abs(r)
            corr[j, i] = abs(r)
    return corr


def process_candidates(arr, vmask, cols, min_rows=DEFAULT_MIN_ROWS,
                       n_splits=DEFAULT_N_SPLITS,
                       rmse_threshold=DEFAULT_RMSE_THRESHOLD, verbose=True):
    """
    Detect index columns via row-residual sparse convex test.

    For each column, greedily add the next most-correlated predictor as long
    as co-observed rows stay >= min_rows. This gives each column the maximum
    K the data supports — no arbitrary cap.

    Returns dict with index_cols, farmer_cols, residual_data, etc.
    """
    D = len(cols)

    if verbose:
        print(f"\n{'='*70}")
        print(f"  PHASE 1: Row-Residual Index Detection (adaptive K, min_rows={min_rows})")
        print(f"{'='*70}\n")
        print(f"  Computing pairwise correlations...")

    corr = pairwise_abs_corr(arr, vmask)

    if verbose:
        print(f"  Running convex residual test ({n_splits} splits per column)...")

    residual_data = []
    for ci in range(D):
        other_corrs = [(j, corr[ci, j]) for j in range(D)
                       if j != ci and not np.isnan(corr[ci, j])]
        other_corrs.sort(key=lambda x: -x[1])

        # Greedily add predictors while co-observed rows >= min_rows
        selected = []
        co_obs = vmask[:, ci].copy()
        for j, _ in other_corrs:
            candidate_obs = co_obs & vmask[:, j]
            if candidate_obs.sum() < min_rows:
                break
            selected.append(j)
            co_obs = candidate_obs

        rows = np.where(co_obs)[0]
        K_used = len(selected)

        if K_used < 2:
            residual_data.append({
                "col": cols[ci], "col_idx": ci,
                "rmse": float("inf"), "std": 0.0,
                "co_obs": int(len(rows)), "K": K_used,
            })
            continue

        y_all = arr[rows, ci]
        X_all = arr[np.ix_(rows, selected)]

        rmses = []
        for _ in range(n_splits):
            idx = np.random.permutation(len(rows))
            half = len(idx) // 2
            tr, te = idx[:half], idx[half:]
            if len(tr) < 10 or len(te) < 10:
                continue
            coefs, _ = fit_convex(y_all[tr], X_all[tr])
            pred = X_all[te] @ coefs
            rmse = np.sqrt(np.mean((y_all[te] - pred) ** 2))
            rmses.append(rmse)

        mean_rmse = float(np.mean(rmses)) if rmses else float("inf")
        std_rmse = float(np.std(rmses)) if rmses else 0.0
        residual_data.append({
            "col": cols[ci], "col_idx": ci,
            "rmse": mean_rmse, "std": std_rmse,
            "co_obs": int(len(rows)), "K": K_used,
        })

    residual_data.sort(key=lambda x: x["rmse"])

    index_cols = []
    index_idxs = []
    for rd in residual_data:
        if rd["rmse"] < rmse_threshold:
            index_cols.append(rd["col"])
            index_idxs.append(rd["col_idx"])

    farmer_idxs = sorted(set(range(D)) - set(index_idxs))
    farmer_cols = [cols[i] for i in farmer_idxs]

    if verbose:
        print(f"\n  {'Rank':>4}  {'Column':<8}  {'RMSE':>10}  {'Std':>8}  {'CoObs':>6}  {'K':>3}")
        print(f"  {'-'*51}")
        for rank, rd in enumerate(residual_data, 1):
            tag = " *" if rd["rmse"] < rmse_threshold else ""
            print(f"  {rank:4d}  {rd['col']:<8}  {rd['rmse']:10.4f}"
                  f"  {rd['std']:8.4f}  {rd['co_obs']:6d}  {rd.get('K',0):3d}{tag}")
        print(f"\n  ┌─────────────────────────────────────────┐")
        print(f"  │  Detected {len(index_cols)} indices (threshold={rmse_threshold})")
        print(f"  │  {index_cols}")
        print(f"  └─────────────────────────────────────────┘")

    return {
        "index_cols": index_cols,
        "index_idxs": index_idxs,
        "farmer_cols": farmer_cols,
        "farmer_idxs": farmer_idxs,
        "residual_data": residual_data,
        "rmse_threshold": rmse_threshold,
    }


def hardcoded_candidates(cols, candidate_nums, verbose=True):
    """Skip detection — use hardcoded column numbers as indices."""
    D = len(cols)
    col_num_map = {int(c.split("_")[1]): i for i, c in enumerate(cols)}

    index_idxs = []
    index_cols = []
    for n in candidate_nums:
        if n not in col_num_map:
            raise ValueError(f"col_{n:02d} not found in columns")
        idx = col_num_map[n]
        index_idxs.append(idx)
        index_cols.append(cols[idx])

    farmer_idxs = sorted(set(range(D)) - set(index_idxs))
    farmer_cols = [cols[i] for i in farmer_idxs]

    if verbose:
        print(f"\n  Hardcoded candidates: {index_cols}")

    return {
        "index_cols": index_cols,
        "index_idxs": index_idxs,
        "farmer_cols": farmer_cols,
        "farmer_idxs": farmer_idxs,
        "residual_data": [],
        "rmse_threshold": 0,
    }


# ── Data availability diagnostic ─────────────────────────────────────────

def print_data_availability(arr, vmask, cols, residual_data, min_rows=20):
    """
    For each candidate, show how co-observation decays as predictors are added.
    Flags columns where the true decomposition might be unverifiable on
    original data (the "col_42 problem").
    """
    D = len(cols)
    corr = pairwise_abs_corr(arr, vmask)

    print(f"\n{'='*70}")
    print(f"  DATA AVAILABILITY DIAGNOSTIC")
    print(f"  (co-observed rows as top-correlated predictors are added)")
    print(f"{'='*70}\n")

    flagged = []
    for rd in residual_data:
        ci = rd["col_idx"]
        col_name = rd["col"]

        other_corrs = [(j, corr[ci, j]) for j in range(D)
                       if j != ci and not np.isnan(corr[ci, j])]
        other_corrs.sort(key=lambda x: -x[1])

        # Track co-obs as we add predictors
        co_obs = vmask[:, ci].copy()
        base_obs = co_obs.sum()
        decay = [(0, base_obs, "-")]
        for k, (j, r) in enumerate(other_corrs, 1):
            co_obs_new = co_obs & vmask[:, j]
            n = co_obs_new.sum()
            decay.append((k, n, cols[j]))
            if n < min_rows:
                break
            co_obs = co_obs_new

        max_k = decay[-1][0] if decay[-1][1] >= min_rows else decay[-2][0]
        drop_rate = 1.0 - (decay[min(5, len(decay)-1)][1] / base_obs) if base_obs > 0 else 0

        # Check for "blind spots": pairs of top-10 correlated columns that
        # have 0 co-obs with the target
        top10 = [j for j, _ in other_corrs[:10]]
        blind_pairs = []
        for i in range(len(top10)):
            for j in range(i+1, len(top10)):
                tri = vmask[:, ci] & vmask[:, top10[i]] & vmask[:, top10[j]]
                if tri.sum() == 0:
                    blind_pairs.append((cols[top10[i]], cols[top10[j]]))

        is_flagged = len(blind_pairs) > 0 or max_k < 5

        print(f"  {col_name}  (obs={base_obs}, max_K={max_k}, "
              f"RMSE={rd['rmse']:.3f})")
        print(f"    K:    ", end="")
        for k, n, _ in decay[:11]:
            print(f" {k:>4}", end="")
        print()
        print(f"    rows: ", end="")
        for k, n, _ in decay[:11]:
            marker = "!" if n < min_rows else " "
            print(f" {n:>3}{marker}", end="")
        print()

        if blind_pairs:
            print(f"    ⚠ BLIND SPOTS: {len(blind_pairs)} top-10 pairs "
                  f"with 0 triple co-obs:")
            for a, b in blind_pairs[:5]:
                print(f"      {col_name} + {a} + {b}: 0 rows")
            if len(blind_pairs) > 5:
                print(f"      ... and {len(blind_pairs)-5} more")
            flagged.append(col_name)
        elif max_k < 5:
            print(f"    ⚠ LOW MAX K: only {max_k} predictors fit "
                  f"with >={min_rows} co-obs")
            flagged.append(col_name)
        print()

    if flagged:
        print(f"  ┌─────────────────────────────────────────────────────┐")
        print(f"  │  ⚠ {len(flagged)} columns with data availability issues:")
        print(f"  │    {flagged}")
        print(f"  │  These may have unverifiable decompositions on")
        print(f"  │  original data (need filled data to prove).")
        print(f"  └─────────────────────────────────────────────────────┘")
    else:
        print(f"  ✓ No data availability issues detected.")


# ── Plotting ──────────────────────────────────────────────────────────────

def plot_residual_ranking(residual_data, rmse_threshold,
                          save_path="analysis/residuals.png"):
    """Horizontal bar chart: all columns ranked by test RMSE."""
    finite = [d for d in residual_data if np.isfinite(d["rmse"])]
    inf_cols = [d for d in residual_data if not np.isfinite(d["rmse"])]
    if not finite:
        return

    n = len(finite)
    fig, ax = plt.subplots(figsize=(11, max(8, n * 0.28)))

    rmses = [d["rmse"] for d in finite]
    labels = [d["col"] for d in finite]
    y_pos = np.arange(n)

    c_index = "#26A69A"
    c_farmer = "#B0BEC5"
    colors = [c_index if r < rmse_threshold else c_farmer for r in rmses]

    bars = ax.barh(y_pos, rmses, color=colors, edgecolor="white",
                   linewidth=0.5, height=0.75)
    ax.axvline(x=rmse_threshold, color="#E53935", linestyle="--",
               linewidth=2, alpha=0.8, label=f"threshold = {rmse_threshold:.1f}")

    k_vals = [d.get("K", 0) for d in finite]
    for i, (rmse, label, k) in enumerate(zip(rmses, labels, k_vals)):
        ax.text(rmse + 0.15, i, f"{rmse:.2f}  (K={k})",
                va="center", fontsize=7, color="#555")

    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=8, fontfamily="monospace")
    ax.invert_yaxis()
    ax.set_xlabel("Test RMSE (convex fit, train/test split)", fontsize=11)
    ax.set_title("Phase 1: Row-Residual Index Detection\n"
                 "Columns ranked by predictability — lower = more likely index",
                 fontsize=13, fontweight="bold")
    ax.legend(loc="lower right", fontsize=11, framealpha=0.9)
    ax.grid(axis="x", alpha=0.2)
    ax.set_xlim(0, max(rmses) * 1.12)

    n_idx = sum(1 for r in rmses if r < rmse_threshold)
    ax.axhspan(-0.5, n_idx - 0.5, alpha=0.06, color=c_index, zorder=0)

    if inf_cols:
        ax.text(0.98, 0.02,
                f"+{len(inf_cols)} cols with insufficient co-obs (not shown)",
                transform=ax.transAxes, ha="right", va="bottom",
                fontsize=8, color="#999")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved {save_path}")


def plot_residual_gaps(residual_data, rmse_threshold,
                       save_path="analysis/gaps.png"):
    """Two-panel: sorted RMSE values + gap ratios between consecutive."""
    finite = [d for d in residual_data if np.isfinite(d["rmse"])]
    if len(finite) < 3:
        return

    rmses = [d["rmse"] for d in finite]
    labels = [d["col"] for d in finite]
    n = len(rmses)
    x = np.arange(n)

    gaps = [rmses[i + 1] / rmses[i] if rmses[i] > 0 else 1.0
            for i in range(n - 1)]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 9),
                                    gridspec_kw={"height_ratios": [3, 2]})

    c_index = "#26A69A"
    c_farmer = "#B0BEC5"
    colors = [c_index if r < rmse_threshold else c_farmer for r in rmses]

    ax1.bar(x, rmses, color=colors, edgecolor="white", linewidth=0.5, width=0.8)
    ax1.axhline(y=rmse_threshold, color="#E53935", linestyle="--",
                linewidth=2, alpha=0.8, label=f"threshold = {rmse_threshold:.1f}")
    ax1.set_ylabel("Test RMSE", fontsize=11)
    ax1.set_title("Phase 1: Sorted RMSE with Gap Analysis", fontsize=13,
                  fontweight="bold")
    ax1.legend(loc="upper left", fontsize=10)
    ax1.grid(axis="y", alpha=0.2)
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, rotation=90, fontsize=7, fontfamily="monospace")

    n_idx = sum(1 for r in rmses if r < rmse_threshold)
    if n_idx > 0 and n_idx < n:
        ax1.axvline(x=n_idx - 0.5, color="#E53935", linewidth=1.5, alpha=0.5)

    gap_x = np.arange(len(gaps))
    gap_colors = ["#FF7043" if g > 1.15 else "#90A4AE" for g in gaps]
    ax2.bar(gap_x, gaps, color=gap_colors, edgecolor="white",
            linewidth=0.5, width=0.8)
    ax2.axhline(y=1.0, color="#999", linewidth=0.5)
    ax2.axhline(y=1.15, color="#FF7043", linestyle=":", linewidth=1.5,
                alpha=0.6, label="1.15x")
    ax2.set_ylabel("Gap ratio (next / current)", fontsize=11)
    ax2.set_xlabel("Rank", fontsize=11)
    ax2.legend(loc="upper left", fontsize=9)
    ax2.grid(axis="y", alpha=0.2)

    gap_labels = [f"{labels[i]}" for i in range(len(gaps))]
    ax2.set_xticks(gap_x)
    ax2.set_xticklabels(gap_labels, rotation=90, fontsize=7, fontfamily="monospace")

    for i, g in enumerate(gaps):
        if g > 1.15:
            ax2.text(i, g + 0.01, f"{g:.2f}", ha="center", fontsize=7,
                     color="#D84315", fontweight="bold")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved {save_path}")


# ── Serialization ─────────────────────────────────────────────────────────

def save_candidates(result, path="intermediates/candidates.json"):
    """Serialize candidates result to JSON."""
    import json, os
    os.makedirs(os.path.dirname(path), exist_ok=True)
    out = {
        "index_cols": result["index_cols"],
        "index_idxs": result["index_idxs"],
        "farmer_cols": result["farmer_cols"],
        "farmer_idxs": result["farmer_idxs"],
        "residual_data": result.get("residual_data", []),
        "rmse_threshold": result.get("rmse_threshold", 0),
    }
    with open(path, "w") as f:
        json.dump(out, f, indent=2)
    print(f"  Saved {path}")


def load_candidates(path="intermediates/candidates.json"):
    """Load candidates result from JSON."""
    import json
    with open(path) as f:
        return json.load(f)


# ── Standalone entry point ────────────────────────────────────────────────

# --- rep2nb: register module so imports resolve ---
import types as _types, sys as _sys
_mod = _sys.modules.get('candidates') or _types.ModuleType('candidates')
for _n in ['np', 'pd', 'matplotlib', 'plt', 'fit_convex', 'DEFAULT_MIN_ROWS', 'DEFAULT_N_SPLITS', 'DEFAULT_RMSE_THRESHOLD', 'pairwise_abs_corr', 'process_candidates', 'hardcoded_candidates', 'print_data_availability', 'plot_residual_ranking', 'plot_residual_gaps', 'save_candidates', 'load_candidates']:
    if _n in globals(): setattr(_mod, _n, globals()[_n])
_sys.modules['candidates'] = _mod
del _types, _sys, _mod


#### `problem-1_2/em.py`

EM pipeline — loads coefficients from intermediates, runs EM loop
(SVD ↔ re-regression) to refine uncertain indices, outputs to answers/.

Run coefficients.py first to generate intermediates/coefficients.json.

In [ ]:
# === problem-1_2/em.py ===

__file__ = 'em.py'
import os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import time, argparse, json

from coefficients import (load_coefficients, fill_from_known,
                          reregress_on_completed, save_coefficients,
                          distribute_through)
from matrix import iterative_svd_complete


EM_COEF_TOL = 1e-4


def verify_rmse(idx_col, cols, decompositions, data):
    """RMSE on co-observed original rows."""
    dec = decompositions[idx_col]
    farmer_names = [cols[fi] for fi in dec["farmer_idxs"]]
    valid = data[[idx_col] + farmer_names].dropna()
    if len(valid) < 3:
        return float("inf")
    pred = valid[farmer_names].values @ dec["coefs"]
    return float(np.sqrt(np.mean((pred - valid[idx_col].values) ** 2)))


def main():
    parser = argparse.ArgumentParser(description="EM refinement pipeline")
    parser.add_argument("--em-iters", type=int, default=10,
                        help="Max EM iterations (default 10)")
    args = parser.parse_args()

    coef_path = "intermediates/coefficients.json"
    if not os.path.exists(coef_path):
        print(f"ERROR: {coef_path} not found. Run coefficients.py first.")
        sys.exit(1)

    os.makedirs("analysis", exist_ok=True)
    t_start = time.time()

    # ── Load data ─────────────────────────────────────────────────────────
    df = pd.read_csv("../data/limestone_data_challenge_2026.data.csv")
    cols = [c for c in df.columns if c.startswith("col_")]
    data = df[cols].copy()
    T, D = data.shape
    print(f"Data: {T}x{D}, NaN rate: {data.isna().mean().mean():.3f}")

    arr = data.values.astype(np.float64)
    vmask = ~np.isnan(arr)

    # ── Load coefficients from intermediate ───────────────────────────────
    coef_data = load_coefficients(coef_path)
    decompositions = coef_data["decompositions"]
    index_cols = coef_data["index_cols"]
    farmer_idxs = coef_data["farmer_idxs"]
    print(f"Loaded coefficients for: {index_cols}")

    # Rebuild filled matrix from the loaded decompositions
    filled, fmask = fill_from_known(arr, vmask, decompositions, cols,
                                     verbose=True)

    # Classify proven vs uncertain
    proven = []
    uncertain = []
    for c in index_cols:
        v = verify_rmse(c, cols, decompositions, data)
        tag = "proven" if v < 0.01 else "uncertain"
        if v < 0.01:
            proven.append(c)
        else:
            uncertain.append(c)
        print(f"  {c}: verify RMSE={v:.6f} [{tag}]")

    print(f"\n  Proven (locked): {proven}")
    print(f"  Uncertain (EM needed): {uncertain}")

    # ── SVD completion ────────────────────────────────────────────────────
    best_rank = len(farmer_idxs)
    obs_pct = fmask.sum() / fmask.size

    print(f"\n{'='*70}")
    print(f"  SVD + EM Refinement (rank={best_rank}, "
          f"warm start {obs_pct:.1%} observed)")
    print(f"{'='*70}\n")

    completed, obs_rmse = iterative_svd_complete(
        filled, fmask, best_rank, max_iter=500, rel_tol=1e-6)
    print(f"  Initial SVD obs RMSE: {obs_rmse:.6f}")

    # ── EM loop ───────────────────────────────────────────────────────────
    em_history = {c: [verify_rmse(c, cols, decompositions, data)]
                  for c in index_cols}
    em_svd_rmse = []
    em_deltas = []

    if uncertain:
        best_verify = {c: verify_rmse(c, cols, decompositions, data)
                       for c in uncertain}
        best_decomp = {c: dict(decompositions[c]) for c in uncertain}

        print(f"\n  EM LOOP: Re-regressing {uncertain}")
        for c in uncertain:
            print(f"    {c}: initial verify={best_verify[c]:.6f}")

        for em_it in range(1, args.em_iters + 1):
            print(f"\n  -- EM iteration {em_it} --")

            prev_coefs = {c: np.array(decompositions[c]["coefs"], dtype=float)
                          for c in uncertain}

            for c in uncertain:
                idx_i = cols.index(c)
                re_fi, re_coefs, re_rmse = reregress_on_completed(
                    idx_i, farmer_idxs, completed, cols, top_k=30)
                if re_coefs is not None:
                    candidate = {
                        "method": "em",
                        "farmer_idxs": re_fi,
                        "coefs": re_coefs,
                    }
                    old_dec = decompositions[c]
                    decompositions[c] = candidate
                    v = verify_rmse(c, cols, decompositions, data)

                    if v < best_verify[c]:
                        best_verify[c] = v
                        best_decomp[c] = dict(candidate)
                        print(f"    {c}: RMSE={re_rmse:.6f}, k={len(re_fi)}, "
                              f"verify={v:.6f} (improved)")
                        for fi, w in sorted(zip(re_fi, re_coefs),
                                            key=lambda x: -x[1]):
                            if w > 0.01:
                                print(f"        {cols[fi]}: {w:.6f}")
                    else:
                        decompositions[c] = old_dec
                        print(f"    {c}: verify={v:.6f} > "
                              f"best {best_verify[c]:.6f}, reverted")

            for c in index_cols:
                em_history[c].append(
                    verify_rmse(c, cols, decompositions, data))

            working = completed.copy()
            for c in index_cols:
                idx_i = cols.index(c)
                dec = decompositions[c]
                working[:, idx_i] = (completed[:, dec["farmer_idxs"]]
                                     @ dec["coefs"])
            working[vmask] = arr[vmask]

            completed, obs_rmse = iterative_svd_complete(
                arr, vmask, best_rank, max_iter=300, rel_tol=1e-6,
                warm_start=working)
            em_svd_rmse.append(obs_rmse)
            print(f"    SVD obs RMSE: {obs_rmse:.6f}")

            max_delta = 0.0
            for c in uncertain:
                new_c = np.array(decompositions[c]["coefs"], dtype=float)
                old_c = prev_coefs[c]
                if len(new_c) == len(old_c):
                    delta = np.max(np.abs(new_c - old_c))
                else:
                    delta = 1.0
                max_delta = max(max_delta, delta)
            em_deltas.append(max_delta)

            print(f"    Max coef delta: {max_delta:.8f}")
            if max_delta < EM_COEF_TOL:
                print(f"\n  EM converged after {em_it} iterations.")
                break
        else:
            print(f"\n  EM reached max iterations ({args.em_iters}).")

        for c in uncertain:
            decompositions[c] = best_decomp[c]
        print(f"\n  EM done — restored best coefficients per column:")
        for c in uncertain:
            print(f"    {c}: best verify={best_verify[c]:.6f}")

        # ── Convergence plot ──────────────────────────────────────────────
        n_em = len(em_deltas)
        if n_em > 0:
            fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
            iters = list(range(n_em))

            for c in uncertain:
                axes[0].plot(iters, em_history[c][1:], "o-", label=c,
                             linewidth=2, markersize=6)
            axes[0].set_ylabel("Verify RMSE")
            axes[0].set_title("EM Convergence")
            axes[0].legend()
            axes[0].grid(True, alpha=0.3)

            axes[1].plot(iters, em_svd_rmse, "s-", color="#2196F3",
                         linewidth=2, markersize=6)
            axes[1].set_ylabel("SVD Obs RMSE")
            axes[1].grid(True, alpha=0.3)

            axes[2].plot(iters, em_deltas, "^-", color="#FF5722",
                         linewidth=2, markersize=6)
            axes[2].axhline(y=EM_COEF_TOL, color="red", linestyle="--",
                             alpha=0.7, label=f"tol={EM_COEF_TOL}")
            axes[2].set_ylabel("Max Coef Delta")
            axes[2].set_xlabel("EM Iteration")
            axes[2].legend()
            axes[2].grid(True, alpha=0.3)

            plt.tight_layout()
            plt.savefig("analysis/em_convergence.png", dpi=150)
            plt.close()
            print(f"  Saved analysis/em_convergence.png")
    else:
        print(f"\n  All indices proven — skipping EM loop.")

    # ── Save EM coefficients ──────────────────────────────────────────────
    accepted_cols = list(decompositions.keys())
    accepted_idxs = [cols.index(c) for c in accepted_cols]
    farmer_idx_set = set(range(D)) - set(accepted_idxs)
    farmer_idxs_final = sorted(farmer_idx_set)
    farmer_cols_final = [cols[i] for i in farmer_idxs_final]

    save_coefficients(
        decompositions,
        accepted_cols, accepted_idxs,
        farmer_cols_final, farmer_idxs_final,
        cols,
        path="intermediates/em_coefficients.json",
    )

    print(f"\nTotal time: {time.time()-t_start:.1f}s")

# --- rep2nb: register module so imports resolve ---
import types as _types, sys as _sys
_mod = _sys.modules.get('em') or _types.ModuleType('em')
for _n in ['os', 'sys', 'np', 'pd', 'matplotlib', 'plt', 'time', 'argparse', 'json', 'load_coefficients', 'fill_from_known', 'reregress_on_completed', 'save_coefficients', 'distribute_through', 'iterative_svd_complete', 'EM_COEF_TOL', 'verify_rmse', 'main']:
    if _n in globals(): setattr(_mod, _n, globals()[_n])
_sys.modules['em'] = _mod
del _types, _sys, _mod


#### `problem-1_2/pipeline.py`

Full pipeline — runs everything from scratch.
candidates → coefficients → SVD → EM → answers/

With --intermediates: also saves intermediates/ and analysis/ plots.
Without: just answers, no plots, no extra IO.

In [ ]:
# === problem-1_2/pipeline.py ===

__file__ = 'pipeline.py'
import sys as _sys; _sys.argv = [_sys.argv[0]]; del _sys
import os
import numpy as np
import pandas as pd
import time, argparse

from candidates import (process_candidates, hardcoded_candidates,
                        DEFAULT_MIN_ROWS, DEFAULT_N_SPLITS, DEFAULT_RMSE_THRESHOLD)
from coefficients import (recover_coefficients, refit_weights,
                          distribute_through)
from matrix import iterative_svd_complete
from trend import fit_all_columns, build_warm_start


ANSWERS_DIR = "../answers"
EM_COEF_TOL = 1e-4


def verify_rmse(idx_col, cols, decompositions, data):
    dec = decompositions[idx_col]
    farmer_names = [cols[fi] for fi in dec["farmer_idxs"]]
    valid = data[[idx_col] + farmer_names].dropna()
    if len(valid) < 3:
        return float("inf")
    pred = valid[farmer_names].values @ dec["coefs"]
    return float(np.sqrt(np.mean((pred - valid[idx_col].values) ** 2)))


def main():
    parser = argparse.ArgumentParser(description="Limestone — full pipeline")
    parser.add_argument("--min-rows", type=int, default=DEFAULT_MIN_ROWS)
    parser.add_argument("--threshold", type=float, default=DEFAULT_RMSE_THRESHOLD)
    parser.add_argument("--em-iters", type=int, default=10)
    parser.add_argument("--intermediates", action="store_true",
                        help="Save intermediates + analysis plots")
    parser.add_argument("--candidates", type=int, nargs="+", default=None,
                        help="Hardcode candidates by col number")
    args = parser.parse_args()

    os.makedirs(ANSWERS_DIR, exist_ok=True)
    if args.intermediates:
        os.makedirs("intermediates", exist_ok=True)
        os.makedirs("analysis", exist_ok=True)

    t_start = time.time()
    np.random.seed(42)

    df = pd.read_csv("../data/limestone_data_challenge_2026.data.csv")
    cols = [c for c in df.columns if c.startswith("col_")]
    data = df[cols].copy()
    T, D = data.shape
    print(f"Data: {T}x{D}, NaN rate: {data.isna().mean().mean():.3f}")

    arr = data.values.astype(np.float64)
    vmask = ~np.isnan(arr)

    # ── Phase 1: candidates ───────────────────────────────────────────────
    if args.candidates:
        result = hardcoded_candidates(cols, args.candidates)
    else:
        result = process_candidates(
            arr, vmask, cols,
            min_rows=args.min_rows, n_splits=DEFAULT_N_SPLITS,
            rmse_threshold=args.threshold,
        )

    if args.intermediates:
        from candidates import (save_candidates, plot_residual_ranking,
                                plot_residual_gaps, print_data_availability)
        save_candidates(result)
        if result["residual_data"]:
            plot_residual_ranking(result["residual_data"],
                                 result["rmse_threshold"])
            plot_residual_gaps(result["residual_data"],
                               result["rmse_threshold"])
            print_data_availability(arr, vmask, cols, result["residual_data"])

    # ── Phase 2: coefficient recovery ─────────────────────────────────────
    decompositions, filled, fmask = recover_coefficients(
        data, arr, vmask, cols,
        result["index_cols"], result["index_idxs"],
        result["farmer_cols"], result["farmer_idxs"],
    )

    # Update index/farmer lists based on what was actually accepted
    accepted_cols = list(decompositions.keys())
    accepted_idxs = [cols.index(c) for c in accepted_cols]
    reclassified = [c for c in result["index_cols"] if c not in decompositions]

    farmer_idx_set = set(range(D)) - set(accepted_idxs)
    farmer_idxs = sorted(farmer_idx_set)
    farmer_cols = [cols[i] for i in farmer_idxs]
    n_farmers = len(farmer_idxs)

    print(f"\n  Final classification: {len(accepted_cols)} indices, {n_farmers} farmers")
    if reclassified:
        print(f"  Reclassified as farmers: {reclassified}")

    # Classify as proven vs uncertain
    proven = []
    uncertain = []
    for c in accepted_cols:
        if decompositions[c].get("proven", False):
            proven.append(c)
        else:
            uncertain.append(c)
    print(f"  Proven: {proven}")
    print(f"  Uncertain (EM needed): {uncertain}")

    if args.intermediates:
        from coefficients import save_coefficients
        save_coefficients(decompositions, accepted_cols, accepted_idxs,
                          farmer_cols, farmer_idxs, cols)

    # ── Phase 2.5: fit trend model per column for warm start ────────────
    t_all = df["time"].values
    print("\n  Fitting triangle-wave trend model per column...")
    trend_fits = fit_all_columns(arr, vmask, t_all, cols, verbose=True)

    trend_warm = build_warm_start(arr, vmask, t_all, cols, trend_fits)

    # Merge: keep coefficient-filled values where available, else use trend
    warm = filled.copy()
    still_nan = np.isnan(warm)
    warm[still_nan] = trend_warm[still_nan]
    warm_mask = fmask | still_nan  # everything is now filled

    # ── Phase 3: SVD completion ───────────────────────────────────────────
    best_rank = n_farmers
    obs_filled = fmask.sum() / fmask.size
    print(f"\n  SVD completion at rank={best_rank} ({n_farmers} farmers), "
          f"warm start from coeff fill ({obs_filled:.1%} observed) "
          f"+ trend model...")

    completed, obs_rmse = iterative_svd_complete(
        arr, vmask, best_rank, warm_start=warm)
    print(f"  SVD obs RMSE={obs_rmse:.6f}")

    if args.intermediates:
        from matrix import save_matrix_result
        save_matrix_result(decompositions, best_rank, obs_rmse, [], cols)

    # ── Phase 4: EM loop ──────────────────────────────────────────────────
    em_history = {c: [verify_rmse(c, cols, decompositions, data)]
                  for c in accepted_cols} if args.intermediates else None
    em_svd_rmse = [] if args.intermediates else None
    em_deltas = [] if args.intermediates else None

    if uncertain:
        fixed_farmers = {c: list(decompositions[c]["farmer_idxs"])
                         for c in uncertain}
        best_verify = {c: verify_rmse(c, cols, decompositions, data)
                       for c in uncertain}
        best_decomp = {c: dict(decompositions[c]) for c in uncertain}

        print(f"\n  EM loop ({args.em_iters} max iters)...")
        for c in uncertain:
            print(f"    {c}: farmers = {[cols[fi] for fi in fixed_farmers[c]]}, "
                  f"initial verify={best_verify[c]:.6f}")

        for em_it in range(1, args.em_iters + 1):
            prev_coefs = {c: np.array(decompositions[c]["coefs"], dtype=float)
                          for c in uncertain}

            for c in uncertain:
                idx_i = cols.index(c)
                re_fi, re_coefs, re_rmse = refit_weights(
                    idx_i, fixed_farmers[c], completed)
                candidate = {
                    "method": "em",
                    "farmer_idxs": re_fi,
                    "coefs": re_coefs,
                    "proven": False,
                }
                old_dec = decompositions[c]
                decompositions[c] = candidate
                v = verify_rmse(c, cols, decompositions, data)

                if v < best_verify[c]:
                    best_verify[c] = v
                    best_decomp[c] = dict(candidate)
                    print(f"    EM {em_it} | {c}: verify={v:.6f} (improved), "
                          f"rmse_completed={re_rmse:.6f}")
                else:
                    decompositions[c] = old_dec
                    print(f"    EM {em_it} | {c}: verify={v:.6f} > "
                          f"best {best_verify[c]:.6f}, reverted")

            if args.intermediates:
                for c in accepted_cols:
                    em_history[c].append(
                        verify_rmse(c, cols, decompositions, data))

            # Reconstruct index columns, then re-SVD
            working = completed.copy()
            for c in accepted_cols:
                idx_i = cols.index(c)
                dec = decompositions[c]
                working[:, idx_i] = (completed[:, dec["farmer_idxs"]]
                                     @ dec["coefs"])
            working[vmask] = arr[vmask]

            completed, obs_rmse = iterative_svd_complete(
                arr, vmask, best_rank, max_iter=300, rel_tol=1e-6,
                warm_start=working)

            max_delta = 0.0
            for c in uncertain:
                new_c = np.array(decompositions[c]["coefs"], dtype=float)
                old_c = prev_coefs[c]
                if len(new_c) == len(old_c):
                    max_delta = max(max_delta, np.max(np.abs(new_c - old_c)))
                else:
                    max_delta = max(max_delta, 1.0)

            if args.intermediates:
                em_svd_rmse.append(obs_rmse)
                em_deltas.append(max_delta)

            print(f"    EM {em_it} | SVD RMSE={obs_rmse:.6f}, "
                  f"coef delta={max_delta:.8f}")
            if max_delta < EM_COEF_TOL:
                print(f"  EM converged at iteration {em_it}.")
                break

        for c in uncertain:
            decompositions[c] = best_decomp[c]
        print(f"\n  EM done — restored best coefficients per column:")
        for c in uncertain:
            print(f"    {c}: best verify={best_verify[c]:.6f}")

    if args.intermediates and em_deltas:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        n_em = len(em_deltas)
        fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
        iters = list(range(n_em))

        for c in uncertain:
            axes[0].plot(iters, em_history[c][1:], "o-", label=c,
                         linewidth=2, markersize=6)
        axes[0].set_ylabel("Verify RMSE")
        axes[0].set_title("EM Convergence")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        axes[1].plot(iters, em_svd_rmse, "s-", color="#2196F3",
                     linewidth=2, markersize=6)
        axes[1].set_ylabel("SVD Obs RMSE")
        axes[1].grid(True, alpha=0.3)

        axes[2].plot(iters, em_deltas, "^-", color="#FF5722",
                     linewidth=2, markersize=6)
        axes[2].axhline(y=EM_COEF_TOL, color="red", linestyle="--",
                         alpha=0.7, label=f"tol={EM_COEF_TOL}")
        axes[2].set_ylabel("Max Coef Delta")
        axes[2].set_xlabel("EM Iteration")
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig("analysis/em_convergence.png", dpi=150)
        plt.close()
        print(f"  Saved analysis/em_convergence.png")

    if args.intermediates:
        from coefficients import save_coefficients as _sc
        _sc(decompositions, accepted_cols, accepted_idxs,
            farmer_cols, farmer_idxs, cols,
            path="intermediates/em_coefficients.json")

    # ── Final reconstruction ──────────────────────────────────────────────
    for c in accepted_cols:
        idx_i = cols.index(c)
        dec = decompositions[c]
        completed[:, idx_i] = (completed[:, dec["farmer_idxs"]]
                               @ dec["coefs"])
    completed[vmask] = arr[vmask]

    # ── Distribute through for problem 1b ─────────────────────────────────
    expanded = distribute_through(decompositions, cols)

    # ── Save answers ──────────────────────────────────────────────────────
    p1a = os.path.join(ANSWERS_DIR, "problem1a_answer.csv")
    p1b = os.path.join(ANSWERS_DIR, "problem1b_answer.csv")
    p2 = os.path.join(ANSWERS_DIR, "problem2_answer.csv")

    # Problem 1a: which columns are indices
    pd.DataFrame({
        "column": farmer_cols + accepted_cols,
        "is_index": [False] * n_farmers + [True] * len(accepted_cols),
    }).to_csv(p1a, index=False)

    # Problem 1b: coefficient matrix (distributed to pure farmers)
    rows_1b = []
    for idx_col in accepted_cols:
        if idx_col in expanded:
            exp = expanded[idx_col]
            for fi, w in zip(exp["farmer_idxs"], exp["coefs"]):
                if w > 1e-4:
                    rows_1b.append({
                        "index_col": idx_col,
                        "constituent_col": cols[fi],
                        "coef": round(float(w), 6),
                    })
        elif idx_col in decompositions:
            dec = decompositions[idx_col]
            for fi, w in zip(dec["farmer_idxs"], dec["coefs"]):
                if w > 1e-4:
                    rows_1b.append({
                        "index_col": idx_col,
                        "constituent_col": cols[fi],
                        "coef": round(float(w), 6),
                    })
    pd.DataFrame(rows_1b).to_csv(p1b, index=False)

    # Problem 2: completed matrix
    out = df.copy()
    for j, c in enumerate(cols):
        out[c] = completed[:, j]
    out.to_csv(p2, index=False)

    print(f"\n  Saved {p1a}, {p1b}, {p2}")
    remaining_nan = np.isnan(completed).sum()
    print(f"  Completed matrix: {remaining_nan} NaN remaining")
    print(f"  Total time: {time.time()-t_start:.1f}s")


main()


In [ ]:
# --- rep2nb: section cleanup ---
import os as _os; _os.chdir('..')
try:
    _os.rmdir('problem-1_2')
except OSError:
    pass
del _os
import sys as _sys
for _k in ['coefficients', 'matrix', 'trend', 'candidates', 'em', 'pipeline']:
    _sys.modules.pop(_k, None)
del _sys, _k

## problem-3_4

In [ ]:
import os as _os; _os.makedirs('problem-3_4', exist_ok=True); _os.chdir('problem-3_4'); del _os

#### `problem-3_4/buy.py`

Problem 3 — Buy 100kg of flour from NaN-priced columns only.

Strategy:
  1. Predict NaN prices (lookup for t<=3649, KNN+LR otherwise).
  2. Buy all 100kg from the cheapest predicted NaN column.

In [ ]:
# === problem-3_4/buy.py ===

__file__ = 'buy.py'
import os, json
import numpy as np
import pandas as pd

DATA_PATH = "../data/limestone_data_challenge_2026.data.csv"
COMPLETED_PATH = "../answers/problem2_answer.csv"
ANSWER_1B = "../answers/problem1b_answer.csv"
COEFF_JSON = "../problem-1_2/intermediates/coefficients.json"

MAX_HISTORICAL_T = 3649
KNN_K = 20
PROJECTION_RANK = 12
KNN_WEIGHT = 0.5

# ── Lazy-loaded state ────────────────────────────────────────────────────────
_completed = None
_cols = None
_decompositions = None  # {idx_col: {"farmer_names": [...], "coefs": [...]}}


def _load_decompositions(cols):
    """Load decompositions from intermediates or answer file."""
    if os.path.exists(COEFF_JSON):
        with open(COEFF_JSON) as f:
            raw = json.load(f)
        raw_dec = raw.get("decompositions", raw)
        decompositions = {}
        for idx_col, dec in raw_dec.items():
            fi_idxs = [int(i) for i in dec["farmer_idxs"]]
            decompositions[idx_col] = {
                "farmer_names": [cols[i] for i in fi_idxs],
                "coefs": [float(c) for c in dec["coefs"]],
            }
        return decompositions

    df = pd.read_csv(ANSWER_1B)
    decompositions = {}
    for idx_col, grp in df.groupby("index_col"):
        decompositions[idx_col] = {
            "farmer_names": grp["constituent_col"].tolist(),
            "coefs": grp["coef"].tolist(),
        }
    return decompositions


def _algebraic_fill(row_vals, cols, decompositions):
    """
    Try to deterministically fill NaN cells from decomposition constraints.
    Returns (known_mask, filled_row) where known[j] = True if observed or algebraic.
    """
    filled = row_vals.copy()
    known = ~np.isnan(filled)

    changed = True
    while changed:
        changed = False
        for idx_col, dec in decompositions.items():
            idx_i = cols.index(idx_col)
            fi_idxs = [cols.index(c) for c in dec["farmer_names"]]
            coefs = np.array(dec["coefs"], dtype=float)

            if known[idx_i]:
                farmer_known = np.array([known[fi] for fi in fi_idxs])
                if (~farmer_known).sum() == 1:
                    j_miss = int(np.where(~farmer_known)[0][0])
                    w = coefs[j_miss]
                    if w >= 1e-8:
                        known_sum = sum(coefs[j] * filled[fi_idxs[j]]
                                        for j in range(len(fi_idxs)) if j != j_miss)
                        filled[fi_idxs[j_miss]] = (filled[idx_i] - known_sum) / w
                        known[fi_idxs[j_miss]] = True
                        changed = True
            else:
                if all(known[fi] for fi in fi_idxs):
                    filled[idx_i] = sum(coefs[j] * filled[fi_idxs[j]]
                                        for j in range(len(fi_idxs)))
                    known[idx_i] = True
                    changed = True

    return known, filled


def _ensure_loaded():
    global _completed, _cols, _decompositions

    if _completed is not None:
        return

    if not os.path.exists(COMPLETED_PATH):
        raise FileNotFoundError(f"Run problem 2 first — need {COMPLETED_PATH}")

    comp_df = pd.read_csv(COMPLETED_PATH)
    _cols = [c for c in comp_df.columns if c != "time"]
    _completed = comp_df[_cols].values.astype(float)
    _decompositions = _load_decompositions(_cols)


def _predict_row(row_vals, t):
    """Return predicted prices for all columns."""
    _ensure_loaded()

    if 0 <= t <= MAX_HISTORICAL_T:
        predicted = _completed[t].copy()
        obs = ~np.isnan(row_vals)
        predicted[obs] = row_vals[obs]
        return predicted

    known, det_filled = _algebraic_fill(row_vals, _cols, _decompositions)
    missing = ~known

    if not missing.any():
        return det_filled

    known_idxs = np.where(known)[0]
    dists = np.sum((_completed[:, known_idxs] - det_filled[known_idxs]) ** 2,
                   axis=1)
    nn_idxs = np.argsort(dists)[:KNN_K]
    weights = 1.0 / (np.sqrt(dists[nn_idxs]) + 1e-8)
    weights /= weights.sum()
    knn_pred = _completed[nn_idxs].T @ weights

    col_means = _completed.mean(axis=0)
    centered = _completed - col_means
    _, s, Vt = np.linalg.svd(centered, full_matrices=False)
    Vt_r = Vt[:PROJECTION_RANK]

    centered_known = det_filled[known_idxs] - col_means[known_idxs]
    V_obs = Vt_r[:, known_idxs].T
    alpha, *_ = np.linalg.lstsq(V_obs, centered_known, rcond=None)
    lr_pred = col_means + Vt_r.T @ alpha

    predicted = KNN_WEIGHT * knn_pred + (1 - KNN_WEIGHT) * lr_pred
    predicted[known] = det_filled[known]
    return predicted


# ── Public trading function ──────────────────────────────────────────────────

def trading_problem_3(row):
    """
    Problem 3 entry point.

    Parameters
    ----------
    row : pd.Series or single-row DataFrame
        One day's bulletin with time + col_00..col_52 (NaNs present).

    Returns
    -------
    pd.DataFrame with columns ``col`` and ``qty`` (ints, sum = 100).
    """
    _ensure_loaded()

    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]

    t = int(row.get("time", -1))
    row_vals = np.array([row.get(c, np.nan) for c in _cols], dtype=float)

    nan_mask = np.isnan(row_vals)
    nan_idxs = np.where(nan_mask)[0]

    if len(nan_idxs) == 0:
        return pd.DataFrame({"col": [_cols[0]], "qty": [100]})

    predicted = _predict_row(row_vals, t)
    prices = predicted[nan_idxs]
    best = nan_idxs[np.argmin(prices)]
    return pd.DataFrame({"col": [_cols[best]], "qty": [100]})


# ── CLI test ─────────────────────────────────────────────────────────────────

def _test_row(t, raw_df, comp_df, cols):
    row = raw_df.iloc[t]
    row_vals = np.array([row[c] for c in cols], dtype=float)

    trades = trading_problem_3(row)
    col = trades["col"].iloc[0]
    qty = trades["qty"].iloc[0]
    true_price = comp_df.loc[t, col]
    cost = qty * true_price

    predicted = _predict_row(row_vals, t)
    pred_price = predicted[cols.index(col)]

    nan_cols = [c for c in cols if np.isnan(row[c])]
    nan_true = [comp_df.loc[t, c] for c in nan_cols]
    best_possible = min(nan_true) * 100
    worst_possible = max(nan_true) * 100

    print(f"  t={t:<5d} buy {qty:3d}kg {col}  "
          f"pred={pred_price:8.2f}  true={true_price:8.2f}  "
          f"err={pred_price - true_price:+7.2f}  "
          f"cost={cost:10.0f}  oracle={best_possible:10.0f}  "
          f"gap={cost - best_possible:+8.0f}  "
          f"NaNs={len(nan_cols)}")

    return cost, best_possible, worst_possible


def _test_synthetic_row(cols, comp_df, base_t=0):
    true_row = comp_df.iloc[base_t]
    row_vals = np.array([true_row[c] for c in cols], dtype=float)

    rng = np.random.default_rng(42)
    mask = rng.random(len(cols)) < 0.5
    synthetic = row_vals.copy()
    synthetic[mask] = np.nan

    row_series = pd.Series({"time": MAX_HISTORICAL_T + 1})
    for i, c in enumerate(cols):
        row_series[c] = synthetic[i]

    trades = trading_problem_3(row_series)
    col = trades["col"].iloc[0]
    qty = trades["qty"].iloc[0]
    ci = cols.index(col)
    true_price = row_vals[ci]
    predicted = _predict_row(synthetic, MAX_HISTORICAL_T + 1)
    pred_price = predicted[ci]

    nan_idxs = np.where(mask)[0]
    best_possible = min(row_vals[i] for i in nan_idxs) * 100
    cost = qty * true_price

    print(f"  t=SYN   buy {qty:3d}kg {col}  "
          f"pred={pred_price:8.2f}  true={true_price:8.2f}  "
          f"err={pred_price - true_price:+7.2f}  "
          f"cost={cost:10.0f}  oracle={best_possible:10.0f}  "
          f"gap={cost - best_possible:+8.0f}  "
          f"NaNs={mask.sum()}")

    return cost, best_possible


import time as _time

raw = pd.read_csv(DATA_PATH)
comp = pd.read_csv(COMPLETED_PATH)
cols = [c for c in raw.columns if c != "time"]

test_ts = [0, 1, 50, 100, 500, 1000, 1825, 2500, 3000, 3649]
print(f"{'='*120}")
print("TEST 1: Historical rows (t <= 3649)")
print(f"{'='*120}")

total_cost, total_oracle = 0.0, 0.0
t1_times = []
for t in test_ts:
    t0 = _time.perf_counter()
    c, o, _ = _test_row(t, raw, comp, cols)
    elapsed = _time.perf_counter() - t0
    t1_times.append(elapsed)
    total_cost += c
    total_oracle += o

print(f"\n  Summary ({len(test_ts)} rows):")
print(f"    Total cost:   {total_cost:>12,.0f}")
print(f"    Oracle cost:  {total_oracle:>12,.0f}")
print(f"    Total gap:    {total_cost - total_oracle:>+12,.0f}")
print(f"    Avg time/row: {1000*sum(t1_times)/len(t1_times):>10.1f} ms")

print(f"\n{'='*120}")
print("TEST 2: Full historical sweep (3650 rows)")
print(f"{'='*120}")

full_cost, full_oracle, full_worst = 0.0, 0.0, 0.0
hits = 0
nan_violations = 0
t2_start = _time.perf_counter()
for t in range(3650):
    row = raw.iloc[t]
    row_vals = np.array([row[c] for c in cols], dtype=float)
    trades = trading_problem_3(row)

    nan_cols_set = {c for c in cols if np.isnan(row[c])}
    for _, trade in trades.iterrows():
        if trade["col"] not in nan_cols_set:
            nan_violations += 1
    assert trades["qty"].sum() == 100

    col_name = trades["col"].iloc[0]
    true_price = comp.loc[t, col_name]
    cost = trades["qty"].iloc[0] * true_price

    nan_true = [comp.loc[t, c] for c in nan_cols_set]
    oracle = min(nan_true) * 100
    worst = max(nan_true) * 100

    if cost <= oracle + 0.01:
        hits += 1
    full_cost += cost
    full_oracle += oracle
    full_worst += worst

t2_elapsed = _time.perf_counter() - t2_start
pct_saved = 100 * (full_worst - full_cost) / (full_worst - full_oracle) if full_worst != full_oracle else 0
print(f"  Total cost:        {full_cost:>14,.0f}")
print(f"  Oracle cost:       {full_oracle:>14,.0f}")
print(f"  Gap vs oracle:     {full_cost - full_oracle:>+14,.0f}")
print(f"  Oracle-hit rate:   {hits}/3650 ({100*hits/3650:.1f}%)")
print(f"  Savings vs worst:  {pct_saved:.1f}%")
print(f"  NaN violations:    {nan_violations}  {'PASS' if nan_violations == 0 else 'FAIL'}")
print(f"  Time:              {t2_elapsed:.2f}s ({1000*t2_elapsed/3650:.1f} ms/row)")

print(f"\n{'='*120}")
print("TEST 3: Synthetic future rows (t > 3649)")
print(f"{'='*120}")

syn_cost, syn_oracle = 0.0, 0.0
t3_times = []
base_rows = [0, 100, 500, 1000, 2000, 3000, 3649]
for bt in base_rows:
    t0 = _time.perf_counter()
    c, o = _test_synthetic_row(cols, comp, base_t=bt)
    elapsed = _time.perf_counter() - t0
    t3_times.append(elapsed)
    syn_cost += c
    syn_oracle += o

print(f"\n  Summary ({len(base_rows)} synthetic rows):")
print(f"    Total cost:   {syn_cost:>12,.0f}")
print(f"    Oracle cost:  {syn_oracle:>12,.0f}")
print(f"    Total gap:    {syn_cost - syn_oracle:>+12,.0f}")
print(f"    Avg time/row: {sum(t3_times)/len(t3_times):>10.2f} s")


#### `problem-3_4/trade.py`

Problem 4 — Arbitrage: buy from NaN src, sell to index dest.

Strategy:
  1. Predict all prices (lookup for t<=3649, KNN+LR otherwise).
  2. Find the cheapest NaN column (src) and most expensive index column (dest).
  3. If dest_price > src_price, trade 100kg for profit.

In [ ]:
# === problem-3_4/trade.py ===

__file__ = 'trade.py'
import os, json
import numpy as np
import pandas as pd

DATA_PATH = "../data/limestone_data_challenge_2026.data.csv"
COMPLETED_PATH = "../answers/problem2_answer.csv"
ANSWER_1A = "../answers/problem1a_answer.csv"
ANSWER_1B = "../answers/problem1b_answer.csv"
COEFF_JSON = "../problem-1_2/intermediates/coefficients.json"

MAX_HISTORICAL_T = 3649
KNN_K = 20
PROJECTION_RANK = 12
KNN_WEIGHT = 0.5

# ── Lazy-loaded state ────────────────────────────────────────────────────────
_completed = None
_cols = None
_decompositions = None
_index_cols = None


def _load_decompositions(cols):
    """Load decompositions from intermediates or answer file."""
    if os.path.exists(COEFF_JSON):
        with open(COEFF_JSON) as f:
            raw = json.load(f)
        raw_dec = raw.get("decompositions", raw)
        decompositions = {}
        for idx_col, dec in raw_dec.items():
            fi_idxs = [int(i) for i in dec["farmer_idxs"]]
            decompositions[idx_col] = {
                "farmer_names": [cols[i] for i in fi_idxs],
                "coefs": [float(c) for c in dec["coefs"]],
            }
        return decompositions

    df = pd.read_csv(ANSWER_1B)
    decompositions = {}
    for idx_col, grp in df.groupby("index_col"):
        decompositions[idx_col] = {
            "farmer_names": grp["constituent_col"].tolist(),
            "coefs": grp["coef"].tolist(),
        }
    return decompositions


def _algebraic_fill(row_vals, cols, decompositions):
    """
    Try to deterministically fill NaN cells from decomposition constraints.
    Returns (known_mask, filled_row).
    """
    filled = row_vals.copy()
    known = ~np.isnan(filled)

    changed = True
    while changed:
        changed = False
        for idx_col, dec in decompositions.items():
            idx_i = cols.index(idx_col)
            fi_idxs = [cols.index(c) for c in dec["farmer_names"]]
            coefs = np.array(dec["coefs"], dtype=float)

            if known[idx_i]:
                farmer_known = np.array([known[fi] for fi in fi_idxs])
                if (~farmer_known).sum() == 1:
                    j_miss = int(np.where(~farmer_known)[0][0])
                    w = coefs[j_miss]
                    if w >= 1e-8:
                        known_sum = sum(coefs[j] * filled[fi_idxs[j]]
                                        for j in range(len(fi_idxs)) if j != j_miss)
                        filled[fi_idxs[j_miss]] = (filled[idx_i] - known_sum) / w
                        known[fi_idxs[j_miss]] = True
                        changed = True
            else:
                if all(known[fi] for fi in fi_idxs):
                    filled[idx_i] = sum(coefs[j] * filled[fi_idxs[j]]
                                        for j in range(len(fi_idxs)))
                    known[idx_i] = True
                    changed = True

    return known, filled


def _ensure_loaded():
    global _completed, _cols, _decompositions, _index_cols

    if _completed is not None:
        return

    if not os.path.exists(COMPLETED_PATH):
        raise FileNotFoundError(f"Run problem 2 first — need {COMPLETED_PATH}")

    comp_df = pd.read_csv(COMPLETED_PATH)
    _cols = [c for c in comp_df.columns if c != "time"]
    _completed = comp_df[_cols].values.astype(float)
    _decompositions = _load_decompositions(_cols)

    a1 = pd.read_csv(ANSWER_1A)
    _index_cols = set(a1.loc[a1["is_index"], "column"].tolist())


def _predict_row(row_vals, t):
    """Return predicted prices for all columns."""
    _ensure_loaded()

    if 0 <= t <= MAX_HISTORICAL_T:
        predicted = _completed[t].copy()
        obs = ~np.isnan(row_vals)
        predicted[obs] = row_vals[obs]
        return predicted

    known, det_filled = _algebraic_fill(row_vals, _cols, _decompositions)
    missing = ~known

    if not missing.any():
        return det_filled

    known_idxs = np.where(known)[0]
    dists = np.sum((_completed[:, known_idxs] - det_filled[known_idxs]) ** 2,
                   axis=1)
    nn_idxs = np.argsort(dists)[:KNN_K]
    weights = 1.0 / (np.sqrt(dists[nn_idxs]) + 1e-8)
    weights /= weights.sum()
    knn_pred = _completed[nn_idxs].T @ weights

    col_means = _completed.mean(axis=0)
    centered = _completed - col_means
    _, s, Vt = np.linalg.svd(centered, full_matrices=False)
    Vt_r = Vt[:PROJECTION_RANK]

    centered_known = det_filled[known_idxs] - col_means[known_idxs]
    V_obs = Vt_r[:, known_idxs].T
    alpha, *_ = np.linalg.lstsq(V_obs, centered_known, rcond=None)
    lr_pred = col_means + Vt_r.T @ alpha

    predicted = KNN_WEIGHT * knn_pred + (1 - KNN_WEIGHT) * lr_pred
    predicted[known] = det_filled[known]
    return predicted


# ── Public trading function ──────────────────────────────────────────────────

def trading_problem_4(row):
    """
    Problem 4 entry point.

    Returns
    -------
    pd.DataFrame with columns ``src_col``, ``dest_col``, ``qty``.
    """
    _ensure_loaded()

    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]

    t = int(row.get("time", -1))
    row_vals = np.array([row.get(c, np.nan) for c in _cols], dtype=float)

    predicted = _predict_row(row_vals, t)
    nan_mask = np.isnan(row_vals)

    src_idxs = np.where(nan_mask)[0]
    if len(src_idxs) == 0:
        return pd.DataFrame({"src_col": [], "dest_col": [], "qty": []})

    dest_idxs = np.array([i for i, c in enumerate(_cols) if c in _index_cols])
    if len(dest_idxs) == 0:
        return pd.DataFrame({"src_col": [], "dest_col": [], "qty": []})

    best_src = src_idxs[np.argmin(predicted[src_idxs])]
    best_dest = dest_idxs[np.argmax(predicted[dest_idxs])]

    src_price = predicted[best_src]
    dest_price = predicted[best_dest]

    if dest_price <= src_price:
        return pd.DataFrame({"src_col": [], "dest_col": [], "qty": []})

    return pd.DataFrame({
        "src_col": [_cols[best_src]],
        "dest_col": [_cols[best_dest]],
        "qty": [100],
    })


# ── CLI test ─────────────────────────────────────────────────────────────────

import time as _time

raw = pd.read_csv(DATA_PATH)
comp = pd.read_csv(COMPLETED_PATH)
cols = [c for c in raw.columns if c != "time"]

_ensure_loaded()
print(f"Index columns: {sorted(_index_cols)}\n")

test_ts = [0, 1, 50, 100, 500, 1000, 1825, 2500, 3000, 3649]
print(f"{'='*130}")
print("TEST 1: Historical rows (t <= 3649)")
print(f"{'='*130}")

total_profit, total_oracle = 0.0, 0.0
for t in test_ts:
    row = raw.iloc[t]
    row_vals = np.array([row[c] for c in cols], dtype=float)
    predicted = _predict_row(row_vals, t)

    trades = trading_problem_4(row)
    nan_mask = np.isnan(row_vals)
    nan_idxs = np.where(nan_mask)[0]
    idx_idxs = np.array([i for i, c in enumerate(cols) if c in _index_cols])

    if len(nan_idxs) > 0 and len(idx_idxs) > 0:
        best_src_price = min(comp.loc[t, cols[i]] for i in nan_idxs)
        best_dest_price = max(comp.loc[t, cols[i]] for i in idx_idxs)
        oracle_profit = max(0, (best_dest_price - best_src_price) * 100)
    else:
        oracle_profit = 0

    if len(trades) == 0:
        profit = 0
        print(f"  t={t:<5d} NO TRADE  oracle={oracle_profit:10.0f}")
    else:
        src = trades["src_col"].iloc[0]
        dest = trades["dest_col"].iloc[0]
        qty = trades["qty"].iloc[0]
        profit = qty * (comp.loc[t, dest] - comp.loc[t, src])
        print(f"  t={t:<5d} {qty:3d}kg {src} → {dest}  "
              f"profit={profit:10.0f}  oracle={oracle_profit:10.0f}  "
              f"gap={profit - oracle_profit:+9.0f}")

    total_profit += profit
    total_oracle += oracle_profit

print(f"\n  Total profit: {total_profit:>12,.0f}  Oracle: {total_oracle:>12,.0f}")

print(f"\n{'='*130}")
print("TEST 2: Full historical sweep (3650 rows)")
print(f"{'='*130}")

full_profit, full_oracle = 0.0, 0.0
n_trades, n_no_trade, violations = 0, 0, 0
t2_start = _time.perf_counter()

for t in range(3650):
    row = raw.iloc[t]
    row_vals = np.array([row[c] for c in cols], dtype=float)
    nan_mask = np.isnan(row_vals)
    nan_idxs = np.where(nan_mask)[0]
    idx_idxs = np.array([i for i, c in enumerate(cols) if c in _index_cols])

    trades = trading_problem_4(row)

    if len(nan_idxs) > 0 and len(idx_idxs) > 0:
        best_src_p = min(comp.loc[t, cols[i]] for i in nan_idxs)
        best_dest_p = max(comp.loc[t, cols[i]] for i in idx_idxs)
        oracle = max(0, (best_dest_p - best_src_p) * 100)
    else:
        oracle = 0

    if len(trades) == 0:
        n_no_trade += 1
    else:
        n_trades += 1
        for _, trade in trades.iterrows():
            if trade["src_col"] not in {cols[i] for i in nan_idxs}:
                violations += 1
            if trade["dest_col"] not in _index_cols:
                violations += 1
        src = trades["src_col"].iloc[0]
        dest = trades["dest_col"].iloc[0]
        qty = trades["qty"].iloc[0]
        full_profit += qty * (comp.loc[t, dest] - comp.loc[t, src])

    full_oracle += oracle

t2_elapsed = _time.perf_counter() - t2_start
print(f"  Total profit:      {full_profit:>14,.0f}")
print(f"  Oracle profit:     {full_oracle:>14,.0f}")
print(f"  Capture rate:      {100*full_profit/full_oracle:.1f}%" if full_oracle > 0 else "")
print(f"  Rows traded:       {n_trades}")
print(f"  Violations:        {violations}  {'PASS' if violations == 0 else 'FAIL'}")
print(f"  Time:              {t2_elapsed:.2f}s ({1000*t2_elapsed/3650:.1f} ms/row)")

print(f"\n{'='*130}")
print("TEST 3: Synthetic future rows (t > 3649)")
print(f"{'='*130}")

idx_idxs = np.array([i for i, c in enumerate(cols) if c in _index_cols])
syn_profit, syn_oracle = 0.0, 0.0
t3_times = []
base_rows = [0, 100, 500, 1000, 2000, 3000, 3649]
for bt in base_rows:
    true_row = comp.iloc[bt]
    true_vals = np.array([true_row[c] for c in cols], dtype=float)

    rng = np.random.default_rng(42)
    mask = rng.random(len(cols)) < 0.5
    synthetic = true_vals.copy()
    synthetic[mask] = np.nan

    row_series = pd.Series({"time": MAX_HISTORICAL_T + 1})
    for i, c in enumerate(cols):
        row_series[c] = synthetic[i]

    t0 = _time.perf_counter()
    trades = trading_problem_4(row_series)
    elapsed = _time.perf_counter() - t0
    t3_times.append(elapsed)

    nan_idxs_syn = np.where(mask)[0]
    best_src_p = min(true_vals[i] for i in nan_idxs_syn)
    best_dest_p = max(true_vals[i] for i in idx_idxs)
    oracle = max(0, (best_dest_p - best_src_p) * 100)

    if len(trades) == 0:
        print(f"  t=SYN(base={bt:<4d}) NO TRADE  oracle={oracle:10.0f}")
    else:
        src = trades["src_col"].iloc[0]
        dest = trades["dest_col"].iloc[0]
        qty = trades["qty"].iloc[0]
        profit = qty * (true_vals[cols.index(dest)] - true_vals[cols.index(src)])
        syn_profit += profit
        print(f"  t=SYN(base={bt:<4d}) {qty:3d}kg {src} → {dest}  "
              f"profit={profit:10.0f}  oracle={oracle:10.0f}  "
              f"gap={profit - oracle:+9.0f}  time={elapsed:.2f}s")

    syn_oracle += oracle

print(f"\n  Total profit: {syn_profit:>12,.0f}  Oracle: {syn_oracle:>12,.0f}")
print(f"  Avg time/row: {sum(t3_times)/len(t3_times):.2f}s")


In [ ]:
# --- rep2nb: section cleanup ---
import os as _os; _os.chdir('..')
try:
    _os.rmdir('problem-3_4')
except OSError:
    pass
del _os
import sys as _sys
for _k in ['buy', 'trade']:
    _sys.modules.pop(_k, None)
del _sys, _k

## problem-5

In [ ]:
import os as _os; _os.makedirs('problem-5', exist_ok=True); _os.chdir('problem-5'); del _os

#### `problem-5/compute_sigma.py`

Compute per-column σ via leave-one-out cross-validation.

For each column j, for ~500 rows where column j is observed:
  - Hide column j
  - Predict it using KNN + low-rank projection from remaining observed columns
  - Record the error

σ_j = std(errors)

Saves result to intermediates/sigma.json.

In [ ]:
# === problem-5/compute_sigma.py ===

__file__ = 'compute_sigma.py'
import os, json
import numpy as np
import pandas as pd

DATA_PATH = "../data/limestone_data_challenge_2026.data.csv"
COMPLETED_PATH = "../answers/problem2_answer.csv"

KNN_K = 20
PROJECTION_RANK = 12
KNN_WEIGHT = 0.5
N_SAMPLES = 500


def main():
    raw = pd.read_csv(DATA_PATH)
    comp = pd.read_csv(COMPLETED_PATH)
    cols = [c for c in raw.columns if c != "time"]
    arr = raw[cols].values.astype(float)
    completed = comp[cols].values.astype(float)
    vmask = ~np.isnan(arr)

    col_means = completed.mean(axis=0)
    centered = completed - col_means
    _, s, Vt = np.linalg.svd(centered, full_matrices=False)
    Vt_r = Vt[:PROJECTION_RANK]

    rng = np.random.default_rng(42)
    sigmas = {}

    for j, col in enumerate(cols):
        obs_rows = np.where(vmask[:, j])[0]
        if len(obs_rows) > N_SAMPLES:
            sample_rows = rng.choice(obs_rows, N_SAMPLES, replace=False)
        else:
            sample_rows = obs_rows

        errors = []
        for row_idx in sample_rows:
            true_val = arr[row_idx, j]
            row_vals = arr[row_idx].copy()
            row_vals[j] = np.nan

            obs = ~np.isnan(row_vals)
            obs_idxs = np.where(obs)[0]

            if len(obs_idxs) < 5:
                continue

            dists = np.sum((completed[:, obs_idxs] - row_vals[obs_idxs]) ** 2, axis=1)
            nn_idxs = np.argsort(dists)[:KNN_K]
            w = 1.0 / (np.sqrt(dists[nn_idxs]) + 1e-8)
            w /= w.sum()
            knn_pred = completed[nn_idxs, j] @ w

            V_obs = Vt_r[:, obs_idxs].T
            centered_obs = row_vals[obs_idxs] - col_means[obs_idxs]
            alpha, *_ = np.linalg.lstsq(V_obs, centered_obs, rcond=None)
            lr_pred = col_means[j] + Vt_r[:, j] @ alpha

            pred = KNN_WEIGHT * knn_pred + (1 - KNN_WEIGHT) * lr_pred
            errors.append(pred - true_val)

        sigma = np.std(errors) if errors else 10.0
        sigmas[col] = round(float(sigma), 4)
        print(f"  {col}: σ = {sigma:.4f}  (n={len(errors)})")

    os.makedirs("intermediates", exist_ok=True)
    with open("intermediates/sigma.json", "w") as f:
        json.dump(sigmas, f, indent=2)
    print(f"\nSaved intermediates/sigma.json")

    avg = np.mean(list(sigmas.values()))
    print(f"Avg σ: {avg:.4f}")

# --- rep2nb: register module so imports resolve ---
import types as _types, sys as _sys
_mod = _sys.modules.get('compute_sigma') or _types.ModuleType('compute_sigma')
for _n in ['os', 'json', 'np', 'pd', 'DATA_PATH', 'COMPLETED_PATH', 'KNN_K', 'PROJECTION_RANK', 'KNN_WEIGHT', 'N_SAMPLES', 'main']:
    if _n in globals(): setattr(_mod, _n, globals()[_n])
_sys.modules['compute_sigma'] = _mod
del _types, _sys, _mod


#### `problem-5/trade.py`

Problem 5 — Limit-order buying with optimal bid pricing.

Single self-contained file. All data loaded from intermediates:
  - Decompositions from problem-1_2/intermediates/coefficients.json
  - Sigma from problem-5/intermediates/sigma.json
  - Classification built on the fly from raw data + decompositions

Score = Σ(qty_i × (median_px - px_i) × I{px_i >= true_px_i})

In [ ]:
# === problem-5/trade.py ===

__file__ = 'trade.py'
import os, json
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.stats import norm

_DIR = os.path.dirname(os.path.abspath(__file__))

DATA_PATH = os.path.join(_DIR, "..", "data",
                         "limestone_data_challenge_2026.data.csv")
COMPLETED_PATH = os.path.join(_DIR, "..", "answers",
                              "problem2_answer.csv")
COEFF_JSON = os.path.join(_DIR, "..", "problem-1_2",
                          "intermediates", "coefficients.json")
ANSWER_1B = os.path.join(_DIR, "..", "answers",
                         "problem1b_answer.csv")
SIGMA_JSON = os.path.join(_DIR, "intermediates", "sigma.json")

MAX_HISTORICAL_T = 3649
KNN_K = 20
PROJECTION_RANK = 12
KNN_WEIGHT = 0.5

SIGMA_ALGEBRAIC = 0.02
SIGMA_P2_CAP = 3.0
SIGMA_OOT_INFLATE = 1.5
DEFAULT_SIGMA = 5.0

# ── Lazy-loaded internal cache ────────────────────────────────────────────────

_state = None


def _load_decompositions(cols):
    """Load raw decompositions (may include index-to-index deps)."""
    if os.path.exists(COEFF_JSON):
        with open(COEFF_JSON) as f:
            raw = json.load(f)
        raw_dec = raw.get("decompositions", raw)
        decompositions = {}
        for idx_col, dec in raw_dec.items():
            fi_idxs = [int(i) for i in dec["farmer_idxs"]]
            decompositions[idx_col] = {
                "farmer_idxs": fi_idxs,
                "farmer_names": [cols[i] for i in fi_idxs],
                "coefs": [float(c) for c in dec["coefs"]],
            }
        return decompositions

    df = pd.read_csv(ANSWER_1B)
    decompositions = {}
    for idx_col, grp in df.groupby("index_col"):
        fi_names = grp["constituent_col"].tolist()
        fi_idxs = [cols.index(c) for c in fi_names]
        decompositions[idx_col] = {
            "farmer_idxs": fi_idxs,
            "farmer_names": fi_names,
            "coefs": grp["coef"].tolist(),
        }
    return decompositions


def _topo_order(decompositions):
    """Topological sort: independent indices first, then dependents."""
    index_set = set(decompositions.keys())
    order = []
    remaining = set(index_set)
    while remaining:
        ready = [c for c in sorted(remaining)
                 if not any(dep in remaining and dep != c
                            for dep in decompositions[c]["farmer_names"]
                            if dep in index_set)]
        if not ready:
            order.extend(sorted(remaining))
            break
        order.extend(ready)
        remaining -= set(ready)
    return order


def _load_sigma():
    """Load per-column sigma from intermediates."""
    if os.path.exists(SIGMA_JSON):
        with open(SIGMA_JSON) as f:
            return json.load(f)
    return {}


def _build_classes(original_arr, vmask, decompositions, cols):
    """Build classification: 0=given, 1=deterministic, 2=imputed."""
    n_rows, n_cols = original_arr.shape
    classes = np.full((n_rows, n_cols), 2, dtype=np.int8)
    classes[vmask] = 0

    filled = original_arr.copy()
    fmask = vmask.copy()

    for _ in range(20):
        new_fills = 0
        for idx_col, dec in decompositions.items():
            idx_i = cols.index(idx_col)
            fi_list = dec["farmer_idxs"]
            coefs = np.array(dec["coefs"], dtype=float)

            for row in np.where(fmask[:, idx_i])[0]:
                farmer_obs = fmask[row, fi_list]
                if (~farmer_obs).sum() != 1:
                    continue
                j_miss = int(np.where(~farmer_obs)[0][0])
                fi_miss = fi_list[j_miss]
                w = coefs[j_miss]
                if w < 1e-8:
                    continue
                known_sum = sum(
                    coefs[j] * filled[row, fi_list[j]]
                    for j in range(len(fi_list)) if j != j_miss
                )
                filled[row, fi_miss] = (filled[row, idx_i] - known_sum) / w
                fmask[row, fi_miss] = True
                classes[row, fi_miss] = 1
                new_fills += 1

            for row in np.where(~fmask[:, idx_i])[0]:
                if fmask[row, fi_list].all():
                    filled[row, idx_i] = sum(
                        coefs[j] * filled[row, fi_list[j]]
                        for j in range(len(fi_list))
                    )
                    fmask[row, idx_i] = True
                    classes[row, idx_i] = 1
                    new_fills += 1

        if new_fills == 0:
            break

    return classes, filled


def _load_all():
    """Load everything needed, cache in module-level _state."""
    global _state
    if _state is not None:
        return _state

    comp_df = pd.read_csv(COMPLETED_PATH)
    cols = [c for c in comp_df.columns if c != "time"]
    completed = comp_df[cols].values.astype(float)

    decompositions = _load_decompositions(cols)
    decomp_order = _topo_order(decompositions)
    sigma = _load_sigma()

    raw_df = pd.read_csv(DATA_PATH)
    raw_arr = raw_df[cols].values.astype(float)
    raw_vmask = ~np.isnan(raw_arr)
    classes_full, _ = _build_classes(raw_arr, raw_vmask, decompositions, cols)

    _state = {
        "cols": cols,
        "completed": completed,
        "decompositions": decompositions,
        "decomp_order": decomp_order,
        "sigma": sigma,
        "classes_full": classes_full,
    }
    return _state


# ── Core helpers ──────────────────────────────────────────────────────────────

def _try_algebraic(row_prices, cols, decompositions, decomp_order):
    """Compute index prices from decompositions in topological order."""
    known = {}
    for c in cols:
        v = row_prices.get(c, np.nan)
        if not np.isnan(v):
            known[c] = v

    results = {}
    for idx_col in decomp_order:
        if idx_col in known:
            continue
        dec = decompositions[idx_col]
        if all(c in known for c in dec["farmer_names"]):
            price = sum(w * known[c]
                        for c, w in zip(dec["farmer_names"], dec["coefs"]))
            known[idx_col] = price
            results[idx_col] = price

    return results


def _optimal_bid(p_hat, sigma, median_est):
    """Find bid maximizing E[profit/kg] = (median - bid) * Phi((bid - p_hat) / sigma)."""
    if sigma < 0.1:
        return round(p_hat + 0.01, 2)

    if p_hat >= median_est:
        return round(p_hat + 0.01, 2)

    def neg_expected_profit(bid):
        fill_prob = norm.cdf((bid - p_hat) / sigma)
        return -(median_est - bid) * fill_prob

    lo = max(p_hat - 2 * sigma, 0.01)
    hi = median_est - 0.01

    if lo >= hi:
        return round(p_hat + 0.01, 2)

    result = minimize_scalar(neg_expected_profit, bounds=(lo, hi),
                             method="bounded")
    return round(result.x, 2)


# ── Public trading function ───────────────────────────────────────────────────

def trading_problem_5(row, _cache=None):
    """
    Problem 5 entry point.

    Parameters
    ----------
    row : pd.Series
        One day's bulletin with 'time' and col_00..col_52 (some NaN).
    _cache : dict or None
        Optional preloaded data for speed. Keys: 'completed', 'cols',
        'classes_full', 'decompositions', 'decomp_order', 'sigma'.
        If None, loads from disk (cached after first call).

    Returns
    -------
    pd.DataFrame with columns: order_col, px, qty
    """
    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]

    if _cache is not None:
        cols = _cache["cols"]
        completed = _cache["completed"]
        decompositions = _cache.get("decompositions", {})
        decomp_order = _cache.get("decomp_order", [])
        sigma_dict = _cache.get("sigma", {})
        classes_full = _cache.get("classes_full", None)
    else:
        s = _load_all()
        cols = s["cols"]
        completed = s["completed"]
        decompositions = s["decompositions"]
        decomp_order = s["decomp_order"]
        sigma_dict = s["sigma"]
        classes_full = s["classes_full"]

    t = int(row.get("time", -1))
    row_vals = np.array([row.get(c, np.nan) for c in cols], dtype=float)
    nan_mask = np.isnan(row_vals)
    nan_idxs = np.where(nan_mask)[0]

    if len(nan_idxs) == 0:
        return pd.DataFrame({"order_col": [cols[0]], "px": [0.01], "qty": [0]})

    predicted = np.full(len(cols), np.nan)
    col_sigma = np.full(len(cols), np.nan)

    obs_mask = ~nan_mask
    predicted[obs_mask] = row_vals[obs_mask]
    col_sigma[obs_mask] = 0.0

    # Path A: historical lookup
    if 0 <= t <= MAX_HISTORICAL_T:
        for j in nan_idxs:
            predicted[j] = completed[t, j]
            if classes_full is not None and classes_full[t, j] == 1:
                col_sigma[j] = SIGMA_ALGEBRAIC
            else:
                col_sigma[j] = min(
                    sigma_dict.get(cols[j], DEFAULT_SIGMA), SIGMA_P2_CAP)

    # Path B: out-of-time prediction
    else:
        row_prices = {cols[j]: row_vals[j] for j in range(len(cols))
                      if not np.isnan(row_vals[j])}
        alg_fills = _try_algebraic(row_prices, cols, decompositions,
                                   decomp_order)

        for col_name, price in alg_fills.items():
            j = cols.index(col_name)
            if nan_mask[j]:
                predicted[j] = price
                col_sigma[j] = SIGMA_ALGEBRAIC

        still_missing = np.where(np.isnan(predicted))[0]
        if len(still_missing) > 0:
            known_idxs = np.where(~np.isnan(predicted))[0]
            known_vals = predicted[known_idxs]

            dists = np.sum((completed[:, known_idxs] - known_vals) ** 2,
                           axis=1)
            nn_idxs = np.argsort(dists)[:KNN_K]
            w = 1.0 / (np.sqrt(dists[nn_idxs]) + 1e-8)
            w /= w.sum()
            knn_pred = completed[nn_idxs].T @ w

            col_means = completed.mean(axis=0)
            centered = completed - col_means
            _, s, Vt = np.linalg.svd(centered, full_matrices=False)
            Vt_r = Vt[:PROJECTION_RANK]
            V_obs = Vt_r[:, known_idxs].T
            centered_known = known_vals - col_means[known_idxs]
            alpha, *_ = np.linalg.lstsq(V_obs, centered_known, rcond=None)
            lr_pred = col_means + Vt_r.T @ alpha

            blended = KNN_WEIGHT * knn_pred + (1 - KNN_WEIGHT) * lr_pred

            for j in still_missing:
                predicted[j] = blended[j]
                col_sigma[j] = sigma_dict.get(cols[j], DEFAULT_SIGMA) \
                               * SIGMA_OOT_INFLATE

    median_est = float(np.median(predicted))

    bids = []
    for j in nan_idxs:
        p_hat = predicted[j]
        sigma = col_sigma[j]
        bid = _optimal_bid(p_hat, sigma, median_est)

        if sigma < 0.1:
            fill_prob = 1.0
        else:
            fill_prob = norm.cdf((bid - p_hat) / sigma)

        exp_profit = (median_est - bid) * fill_prob
        bids.append((j, cols[j], bid, exp_profit, fill_prob))

    profitable = [(j, col, bid, ep, fp)
                  for j, col, bid, ep, fp in bids if ep > 0]

    if not profitable:
        best = max(bids, key=lambda x: x[3])
        return pd.DataFrame({
            "order_col": [best[1]], "px": [best[2]], "qty": [0]
        })

    profitable.sort(key=lambda x: -x[3])
    is_historical = (0 <= t <= MAX_HISTORICAL_T)

    if is_historical:
        top = profitable[0]
        return pd.DataFrame({
            "order_col": [top[1]], "px": [top[2]], "qty": [100]
        })
    else:
        top_n = profitable[:min(3, len(profitable))]
        total_ep = sum(x[3] for x in top_n)

        orders = []
        remaining = 100
        for i, (j, col, bid, ep, fp) in enumerate(top_n):
            if i == len(top_n) - 1:
                qty = remaining
            else:
                qty = max(1, int(round(100 * ep / total_ep)))
                qty = min(qty, remaining)
            remaining -= qty
            if qty > 0:
                orders.append({"order_col": col, "px": bid, "qty": qty})

        if not orders:
            top = profitable[0]
            orders = [{"order_col": top[1], "px": top[2], "qty": 100}]

        return pd.DataFrame(orders)


# ── Backtest ──────────────────────────────────────────────────────────────────

# --- rep2nb: register module so imports resolve ---
import types as _types, sys as _sys
_mod = _sys.modules.get('trade') or _types.ModuleType('trade')
for _n in ['os', 'json', 'np', 'pd', 'minimize_scalar', 'norm', '_DIR', 'DATA_PATH', 'COMPLETED_PATH', 'COEFF_JSON', 'ANSWER_1B', 'SIGMA_JSON', 'MAX_HISTORICAL_T', 'KNN_K', 'PROJECTION_RANK', 'KNN_WEIGHT', 'SIGMA_ALGEBRAIC', 'SIGMA_P2_CAP', 'SIGMA_OOT_INFLATE', 'DEFAULT_SIGMA', '_state', '_load_decompositions', '_topo_order', '_load_sigma', '_build_classes', '_load_all', '_try_algebraic', '_optimal_bid', 'trading_problem_5']:
    if _n in globals(): setattr(_mod, _n, globals()[_n])
_sys.modules['trade'] = _mod
del _types, _sys, _mod


#### `problem-5/pipeline.py`

Problem 5 pipeline — compute σ, build cache, run backtest.

Without --intermediates: computes σ in memory, runs backtest directly.
With --intermediates:    also saves intermediates/sigma.json to disk.

Prerequisites: run problem-1_2/pipeline.py first (needs answers/ and
               problem-1_2/intermediates/coefficients.json).

In [ ]:
# === problem-5/pipeline.py ===

__file__ = 'pipeline.py'
import sys as _sys; _sys.argv = [_sys.argv[0]]; del _sys
import os, sys, json, time, argparse
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(__file__))
from trade import (trading_problem_5, _load_decompositions, _topo_order,
                   _build_classes, DEFAULT_SIGMA)

DATA_PATH = os.path.join(os.path.dirname(__file__), "..", "data",
                         "limestone_data_challenge_2026.data.csv")
COMPLETED_PATH = os.path.join(os.path.dirname(__file__), "..", "answers",
                              "problem2_answer.csv")

KNN_K = 20
PROJECTION_RANK = 12
KNN_WEIGHT = 0.5
N_SAMPLES = 500


def compute_sigma(arr, completed, vmask, cols, verbose=True):
    """LOO cross-validation: per-column prediction error σ."""
    col_means = completed.mean(axis=0)
    centered = completed - col_means
    _, s, Vt = np.linalg.svd(centered, full_matrices=False)
    Vt_r = Vt[:PROJECTION_RANK]

    rng = np.random.default_rng(42)
    sigmas = {}

    for j, col in enumerate(cols):
        obs_rows = np.where(vmask[:, j])[0]
        if len(obs_rows) > N_SAMPLES:
            sample_rows = rng.choice(obs_rows, N_SAMPLES, replace=False)
        else:
            sample_rows = obs_rows

        errors = []
        for row_idx in sample_rows:
            true_val = arr[row_idx, j]
            row_vals = arr[row_idx].copy()
            row_vals[j] = np.nan

            obs = ~np.isnan(row_vals)
            obs_idxs = np.where(obs)[0]
            if len(obs_idxs) < 5:
                continue

            dists = np.sum((completed[:, obs_idxs] - row_vals[obs_idxs]) ** 2,
                           axis=1)
            nn_idxs = np.argsort(dists)[:KNN_K]
            w = 1.0 / (np.sqrt(dists[nn_idxs]) + 1e-8)
            w /= w.sum()
            knn_pred = completed[nn_idxs, j] @ w

            V_obs = Vt_r[:, obs_idxs].T
            centered_obs = row_vals[obs_idxs] - col_means[obs_idxs]
            alpha, *_ = np.linalg.lstsq(V_obs, centered_obs, rcond=None)
            lr_pred = col_means[j] + Vt_r[:, j] @ alpha

            pred = KNN_WEIGHT * knn_pred + (1 - KNN_WEIGHT) * lr_pred
            errors.append(pred - true_val)

        sigma = np.std(errors) if errors else 10.0
        sigmas[col] = round(float(sigma), 4)
        if verbose:
            print(f"  {col}: σ = {sigma:.4f}  (n={len(errors)})")

    return sigmas


def backtest(raw_df, cols, completed, cache, n_rows=500):
    """Run backtest on training rows, return summary dict."""
    rng = np.random.default_rng(42)
    test_rows = sorted(rng.choice(3650, n_rows, replace=False))

    total_score = 0.0
    total_fills = 0
    total_orders = 0
    total_qty_filled = 0
    col_selected = {}
    t0 = time.time()

    for i, t in enumerate(test_rows):
        row = raw_df.iloc[t]
        row_vals = np.array([row[c] for c in cols], dtype=float)
        nan_idxs = np.where(np.isnan(row_vals))[0]

        result = trading_problem_5(row, _cache=cache)

        true_prices = completed[t]
        true_median = float(np.median(true_prices))

        for _, order in result.iterrows():
            col_name = order["order_col"]
            bid = order["px"]
            qty = int(order["qty"])
            j = cols.index(col_name)

            total_orders += 1
            col_selected[col_name] = col_selected.get(col_name, 0) + 1

            if bid >= true_prices[j]:
                total_score += qty * (true_median - bid)
                total_fills += 1
                total_qty_filled += qty

        if (i + 1) % 100 == 0:
            elapsed = time.time() - t0
            print(f"  [{i+1:3d}/{n_rows}] "
                  f"score={total_score:>10.0f}  "
                  f"fills={total_fills}/{total_orders}  "
                  f"time={elapsed:.1f}s")

    elapsed = time.time() - t0
    fill_rate = total_fills / max(total_orders, 1) * 100

    return {
        "total_score": total_score,
        "avg_profit": total_score / n_rows,
        "fill_rate": fill_rate,
        "total_fills": total_fills,
        "total_orders": total_orders,
        "total_qty_filled": total_qty_filled,
        "elapsed": elapsed,
        "col_selected": col_selected,
    }


def main():
    parser = argparse.ArgumentParser(description="Problem 5 pipeline")
    parser.add_argument("--intermediates", action="store_true",
                        help="Save intermediates/sigma.json to disk")
    parser.add_argument("--backtest-rows", type=int, default=500,
                        help="Number of rows to backtest (default 500)")
    args = parser.parse_args()

    t_start = time.time()

    if not os.path.exists(COMPLETED_PATH):
        print(f"ERROR: {COMPLETED_PATH} not found. "
              f"Run problem-1_2/pipeline.py first.")
        sys.exit(1)

    print(f"{'='*70}")
    print(f"  Problem 5 Pipeline")
    print(f"{'='*70}\n")

    # ── Load data ─────────────────────────────────────────────────────────
    raw_df = pd.read_csv(DATA_PATH)
    comp_df = pd.read_csv(COMPLETED_PATH)
    cols = [c for c in raw_df.columns if c != "time"]
    arr = raw_df[cols].values.astype(float)
    completed = comp_df[cols].values.astype(float)
    vmask = ~np.isnan(arr)

    print(f"  Data: {arr.shape[0]}x{arr.shape[1]}, "
          f"NaN rate: {(~vmask).mean():.3f}")

    # ── Step 1: compute per-column σ ──────────────────────────────────────
    print(f"\n  Computing per-column σ (LOO CV, {N_SAMPLES} samples/col)...")
    sigmas = compute_sigma(arr, completed, vmask, cols)
    avg_sigma = np.mean(list(sigmas.values()))
    print(f"\n  Avg σ: {avg_sigma:.4f}")

    if args.intermediates:
        os.makedirs("intermediates", exist_ok=True)
        with open("intermediates/sigma.json", "w") as f:
            json.dump(sigmas, f, indent=2)
        print(f"  Saved intermediates/sigma.json")

    # ── Step 2: build cache ───────────────────────────────────────────────
    print(f"\n  Building classification matrix...")
    decompositions = _load_decompositions(cols)
    decomp_order = _topo_order(decompositions)
    classes_full, _ = _build_classes(arr, vmask, decompositions, cols)

    n_given = (classes_full == 0).sum()
    n_det = (classes_full == 1).sum()
    n_imp = (classes_full == 2).sum()
    print(f"  Classification: {n_given:,} given, {n_det:,} deterministic, "
          f"{n_imp:,} imputed")

    cache = {
        "cols": cols,
        "completed": completed,
        "decompositions": decompositions,
        "decomp_order": decomp_order,
        "sigma": sigmas,
        "classes_full": classes_full,
    }

    # ── Step 3: backtest ──────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"  Backtest: {args.backtest_rows} training rows")
    print(f"{'='*70}\n")

    results = backtest(raw_df, cols, completed, cache,
                       n_rows=args.backtest_rows)

    print(f"\n{'='*70}")
    print(f"  Results")
    print(f"{'='*70}")
    print(f"  Total score (profit):  {results['total_score']:>12.0f}")
    print(f"  Avg profit / day:      {results['avg_profit']:>12.1f}")
    print(f"  Fill rate:             {results['total_fills']}/"
          f"{results['total_orders']} "
          f"({results['fill_rate']:.1f}%)")
    print(f"  Total qty filled:      {results['total_qty_filled']}")
    print(f"  Backtest time:         {results['elapsed']:.1f}s")

    print(f"\n  Top 10 selected columns:")
    for col, cnt in sorted(results["col_selected"].items(),
                           key=lambda x: -x[1])[:10]:
        s = sigmas.get(col, DEFAULT_SIGMA)
        print(f"    {col}: {cnt} times (σ={s:.2f})")

    print(f"\n  Total pipeline time: {time.time() - t_start:.1f}s")


main()


In [ ]:
# --- rep2nb: section cleanup ---
import os as _os; _os.chdir('..')
try:
    _os.rmdir('problem-5')
except OSError:
    pass
del _os